# [Baseline] Task 3 · 다이나믹 프라이싱을 통한 입장 수익 최대화 제안서 작성

**수요 배수 구조**
$$M_t = M_0 \cdot \exp\!\left(\sum_j \delta_j V_{j,t}\right)$$

**주의**: `gamma`/`u` 파라미터는 최적해가 가격 상·하한에 붙지 않도록(내부해가 나오도록) 재보정되었다.

In [31]:
# =========================================================
# 0. import
# =========================================================
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [32]:
# =========================================================
# 1. [출제자 고정 영역] 2024 시즌 잠재관중 M0 테이블
#    - 본 baseline은 2024 시즌 단일 시즌만 사용한다
#    - 2025 시즌은 Phase 1의 정답(hold-out) 구간이므로 별도 관리
# =========================================================
M0_df = pd.DataFrame({
    'team':   ['Panthers', 'Chiefs', 'Rams', 'Titans', 'Dolphins',
               'Jaguars', 'Falcons', 'Chargers', 'Bengals', 'Ravens'],
    'season': 2024,
    'M0':     [15012, 12163, 20471, 10105, 17883, 17575, 17441, 18550, 10544, 12370],
})

print('M0 테이블:', M0_df.shape, '= 10팀 x 1시즌 (2024)')
M0_df.head()

M0 테이블: (10, 3) = 10팀 x 1시즌 (2024)


,team,season,M0
0,Panthers,2024,15012
1,Chiefs,2024,12163
2,Rams,2024,20471
3,Titans,2024,10105
4,Dolphins,2024,17883


In [33]:
# =========================================================
# 2. [출제자 고정 영역] 2024 시즌 구장 수용인원 테이블
# =========================================================
CAPACITY_BASE = {
    'Panthers': 20500, 'Chiefs': 18700, 'Rams': 23750, 'Titans': 17900, 'Dolphins': 23000, 'Jaguars': 23750, 'Falcons': 22900, 'Chargers': 24000, 'Bengals': 16000, 'Ravens': 12000}

CAPACITY_df = pd.DataFrame(
    [{'team': tm, 'season': 2024, 'capacity': cap} for tm, cap in CAPACITY_BASE.items()]
)

print('Capacity 테이블:', CAPACITY_df.shape, '= 10팀 x 1시즌 (2024)')
CAPACITY_df.head()

Capacity 테이블: (10, 3) = 10팀 x 1시즌 (2024)


,team,season,capacity
0,Panthers,2024,20500
1,Chiefs,2024,18700
2,Rams,2024,23750
3,Titans,2024,17900
4,Dolphins,2024,23000


In [34]:
# ==============================================================================
# 3. [출제자 고정 영역] 글로벌 설정 및 구조 파라미터
# ==============================================================================
CONFIG = {
    'gamma_G': 0.00018,   # 일반석 가격 민감도
    'gamma_P': 0.00011,   # 프리미엄석 가격 민감도
    'u_G': 3.0,           # 일반석 기본 효용
    'u_P': 3.7,           # 프리미엄석 기본 효용
    'alpha_G': 0.3,       # 일반석 잔여재고 감쇠계수
    'alpha_P': 0.1,       # 프리미엄석 잔여재고 감쇠계수
    'prop_G': 0.85,       # 구장 내 일반석 비율
    'prop_P': 0.15,       # 구장 내 프리미엄석 비율
    'share_D7': 0.4,      # D-7 시점 시장 배분 비율
    'share_D3': 0.6,      # D-3 시점 시장 배분 비율
    'p_min_ratio': 0.7,   # 최소 가격 (기본가 대비)
    'p_max_ratio': 1.5,   # 최대 가격 (기본가 대비)
}

BASE_PRICES = {'일반석': 12000, '프리미엄석': 35000}

# log_shock(가상지표 합)의 상하한
SHOCK_CLIP = (-0.6, 0.8)

In [35]:
# ==============================================================================
# 4. [출제자 고정 영역] 방법론 데모용 예시 데이터셋 (보고서 4.2절 재현용)
#
#  주의: 이 데이터셋은 방법론을 보여주기 위한 예시일 뿐, 실제 Phase 3 문제 시나리오가 아니다.
#        실제 문제 시나리오는 df_games_scenarios 를 참고할 것.
# ==============================================================================
df_games_demo = pd.DataFrame([
    # 라이벌전 (Jaguars vs Rams)
    {'date': '2024-05-11', 'season': 2024, 'home_team': 'Jaguars', 'away_team': 'Rams', 'day_of_week': '토', 'home_rank': 2, 'away_rank': 3},
    # 평상시 대조군 (Jaguars vs Falcons)
    {'date': '2024-05-18', 'season': 2024, 'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토', 'home_rank': 2, 'away_rank': 5},
])
df_games_demo

,date,season,home_team,away_team,day_of_week,home_rank,away_rank
0,2024-05-11,2024,Jaguars,Rams,토,2,3
1,2024-05-18,2024,Jaguars,Falcons,토,2,5


In [36]:
# ==============================================================================
# 5. [출제자 고정 영역] 핵심 수요 모델 (Multinomial Logit)
# ==============================================================================
def multinomial_logit_demand(M_period, p_G, p_P, gamma_eff_G, gamma_eff_P, cfg):
    """
    출제 가이드라인에 정의된 다항 로짓(Multinomial Logit) 수요 모형 수식을 연산다.

    Parameters:
    - M_period: 해당 판매 기간(D-7 혹은 D-3)에 할당된 시장 수요 규모
    - p_G, p_P: 일반석 및 프리미엄석 가격
    - gamma_eff_G, gamma_eff_P: 희소성이 반영된 유효 가격 민감도

    Returns:
    - Q_G, Q_P: 일반석 및 프리미엄석의 수요 기댓값 (물리적 한도 적용 전)
    """
    v_G = cfg['u_G'] - gamma_eff_G * p_G
    v_P = cfg['u_P'] - gamma_eff_P * p_P

    denom = 1.0 + np.exp(v_G) + np.exp(v_P)

    Q_G = M_period * np.exp(v_G) / denom
    Q_P = M_period * np.exp(v_P) / denom
    return Q_G, Q_P

In [37]:
# ==============================================================================
# 6. [출제자 고정 영역] 기대수익 목적함수
# ==============================================================================
def simulate_sale(prices, M_t, cap_total, cfg=CONFIG):
    p_D7_G, p_D7_P, p_D3_G, p_D3_P = prices

    cap_G = cap_total * cfg['prop_G']
    cap_P = cap_total * cfg['prop_P']

    # ---------- [1단계] D-7 시점 ----------
    f_G_D7 = 1.0
    f_P_D7 = 1.0
    gamma_eff_G_D7 = cfg['gamma_G'] / f_G_D7
    gamma_eff_P_D7 = cfg['gamma_P'] / f_P_D7

    M_D7 = M_t * cfg['share_D7']
    Q_D7_G_pred, Q_D7_P_pred = multinomial_logit_demand(
        M_D7, p_D7_G, p_D7_P, gamma_eff_G_D7, gamma_eff_P_D7, cfg)

    Q_D7_G = min(Q_D7_G_pred, cap_G)
    Q_D7_P = min(Q_D7_P_pred, cap_P)

    stock_ratio_G = (cap_G - Q_D7_G) / cap_G if cap_G > 0 else 0.0
    stock_ratio_P = (cap_P - Q_D7_P) / cap_P if cap_P > 0 else 0.0

    f_G_D3 = 1.0 + cfg['alpha_G'] * (1.0 - stock_ratio_G)
    f_P_D3 = 1.0 + cfg['alpha_P'] * (1.0 - stock_ratio_P)

    gamma_eff_G_D3 = cfg['gamma_G'] / f_G_D3
    gamma_eff_P_D3 = cfg['gamma_P'] / f_P_D3

    M_D3 = M_t * cfg['share_D3']
    Q_D3_G_pred, Q_D3_P_pred = multinomial_logit_demand(
        M_D3, p_D3_G, p_D3_P, gamma_eff_G_D3, gamma_eff_P_D3, cfg)

    Q_D3_G = min(Q_D3_G_pred, cap_G - Q_D7_G)
    Q_D3_P = min(Q_D3_P_pred, cap_P - Q_D7_P)

    revenue = (p_D7_G * Q_D7_G + p_D7_P * Q_D7_P +
               p_D3_G * Q_D3_G + p_D3_P * Q_D3_P)

    return {
        # 수익
        'revenue': revenue,
        # 시장 배분
        'M_D7': M_D7, 'M_D3': M_D3,
        # 좌석 용량
        'cap_G': cap_G, 'cap_P': cap_P,
        # 시점별 판매량
        'Q_D7_G': Q_D7_G, 'Q_D7_P': Q_D7_P,
        'Q_D3_G': Q_D3_G, 'Q_D3_P': Q_D3_P,
        'total_audience': Q_D7_G + Q_D7_P + Q_D3_G + Q_D3_P,
        # 희소성 메커니즘 (D-7 판매 결과 -> D-3 민감도)
        'r_G': stock_ratio_G, 'r_P': stock_ratio_P,      # 잔여재고율
        'f_G': f_G_D3, 'f_P': f_P_D3,                    # 희소성 계수
        'gamma_eff_G_D3': gamma_eff_G_D3,                # 유효 가격민감도
        'gamma_eff_P_D3': gamma_eff_P_D3,
    }


def expected_revenue_objective(prices, M_t, cap_total, cfg):
    """
    최적화 엔진(minimize)에 넘기는 목적함수. 음의 총 기대수익을 반환한다.
    """
    return -simulate_sale(prices, M_t, cap_total, cfg)['revenue']


def revenue_detail(prices, M_t, cap_total, cfg=CONFIG):
    """
    진단/보고서용. simulate_sale 의 결과를 그대로 돌려준다.
    (최적화에는 관여하지 않으며, 목적함수와 완전히 동일한 계산 결과를 보장한다.)
    """
    return simulate_sale(prices, M_t, cap_total, cfg)

In [38]:
# ==============================================================================
# 7. [출제자 고정 영역] SLSQP 최적화 엔진
# ==============================================================================
def optimize_single_game(M_t, cap_total, cfg=CONFIG, multi_start=True):
    bG, bP = BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']
    lo, hi = cfg['p_min_ratio'], cfg['p_max_ratio']

    bounds = [(bG * lo, bG * hi), (bP * lo, bP * hi),
              (bG * lo, bG * hi), (bP * lo, bP * hi)]

    constraints = [
        {'type': 'ineq', 'fun': lambda x: x[1] - x[0]},   # C1: p_D7_P >= p_D7_G
        {'type': 'ineq', 'fun': lambda x: x[3] - x[2]},   # C1: p_D3_P >= p_D3_G
    ]

    starts = [[bG, bP, bG, bP]]
    if multi_start:
        starts += [[bG * 1.2, bP * 1.2, bG * 1.3, bP * 1.3],
                   [bG * 0.8, bP * 0.8, bG * 0.9, bP * 0.9],
                   [bG * 0.5, bP * 0.5, bG * 0.4, bP * 0.4]]

    best = None
    for x0 in starts:
        res = minimize(expected_revenue_objective, x0,
                       args=(M_t, cap_total, cfg),
                       method='SLSQP', bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': 500})
        if res.success and (best is None or res.fun < best.fun):
            best = res
    return best if best is not None else res

In [39]:
# ==============================================================================
# 8. [출제자 고정 영역] 시뮬레이션용 통합 실행 파이프라인
# ==============================================================================
RIVALRY_PAIRS = [
    ('Jaguars', 'Rams'),   # 전통의 라이벌 매치업 (연고지 라이벌)
]

def my_virtual_indicators(row, cfg=CONFIG):
    log_shock = 0.0

    # ---- [지표 1] 라이벌 매치업 효과 (선별적용) ----
    # delta_rival = 0.35  ->  exp(0.35) ≈ 1.42  (수요 약 42% 상승)
    pair = (row['home_team'], row['away_team'])
    if pair in RIVALRY_PAIRS or pair[::-1] in RIVALRY_PAIRS:
        delta_rival = 0.35
        log_shock += delta_rival

    return log_shock

def run_evaluation_pipeline(df, M0_df, CAPACITY_df, indicator_fn = my_virtual_indicators,
                             shock_clip=SHOCK_CLIP, verbose_clip=True):
    d = df.copy()
    d = d.merge(M0_df.rename(columns={'team': 'home_team'}),
                on=['home_team', 'season'], how='left')
    d = d.merge(CAPACITY_df.rename(columns={'team': 'home_team'}),
                on=['home_team', 'season'], how='left')

    results = []
    for _, row in d.iterrows():
        raw_log_shock = float(indicator_fn(row))
        log_shock = float(np.clip(raw_log_shock, *shock_clip))
        is_clipped = abs(raw_log_shock - log_shock) > 1e-9
        M_t = row['M0'] * np.exp(log_shock)

        res = optimize_single_game(M_t, row['capacity'])
        det = revenue_detail(res.x, M_t, row['capacity'])

        _lo = CONFIG['p_min_ratio']; _hi = CONFIG['p_max_ratio']
        _bases = [BASE_PRICES['일반석'], BASE_PRICES['프리미엄석'],
                  BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']]
        corner_hits = sum(
            1 for _p, _b in zip(res.x, _bases)
            if (_p / _b) >= _hi - 5e-3 or (_p / _b) <= _lo + 5e-3
        )

        results.append({
            'game_id':        f"{row['home_team']} vs {row['away_team']}",
            'M0':              row['M0'],
            'raw_log_shock':   round(raw_log_shock, 4),
            'log_shock':       round(log_shock, 4),
            'clipped':         is_clipped,
            'shock_x':         round(np.exp(log_shock), 3),
            'M_t':             round(M_t),
            'capacity':        row['capacity'],
            'p_D7_G':    round(res.x[0]), 'p_D7_P': round(res.x[1]),
            'p_D3_G':    round(res.x[2]), 'p_D3_P': round(res.x[3]),
            'audience':  round(det['total_audience']),
            'revenue':   round(det['revenue']),
            'corner_hits': corner_hits,
            'success':   res.success,
        })
    out = pd.DataFrame(results)

    if verbose_clip:
        clipped_rows = out[out['clipped']]
        if len(clipped_rows) > 0:
            lo, hi = shock_clip
            print(f"[SHOCK_CLIP 경고] {len(clipped_rows)}개 경기에서 log_shock이 "
                  f"허용 범위 ({lo}, {hi})를 벗어나 잘렸습니다:")
            for _, r in clipped_rows.iterrows():
                print(f"   - {r['game_id']:<24} 원래 합 δ = {r['raw_log_shock']:>7.4f}  "
                      f"->  적용된 값 = {r['log_shock']:>6.2f}")
            print("   지표 계수를 조정해 이 범위 안에서 의도한 효과가 나도록 다시 설계해 보세요.\n")

        corner_rows = out[out['corner_hits'] > 0]
        if len(corner_rows) > 0:
            print(f"[코너해 경고] {len(corner_rows)}개 경기에서 최적가가 가격 경계(0.7x/1.5x)에 "
                  f"붙었습니다 (corner_hits > 0):")
            for _, r in corner_rows.iterrows():
                print(f"   - {r['game_id']:<24} corner_hits = {int(r['corner_hits'])}  "
                      f"(log_shock = {r['log_shock']:>6.2f})")
            print("   이 행들은 '더 올리고/내리고 싶은데 경계에 막힌' 코너해라 진짜 내부 최적해가\n"
                  "   아닙니다. 대조군-처치군 비교가 왜곡되므로 지표(δ) 설계를 재검토하세요.\n")

    return out

In [40]:
# ==============================================================================
# 9. [실행] 시나리오 시뮬레이션
# ==============================================================================
res = run_evaluation_pipeline(df_games_demo, M0_df, CAPACITY_df)

bG, bP = BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']
view = res.copy()
for c, b in [('p_D7_G', bG), ('p_D7_P', bP), ('p_D3_G', bG), ('p_D3_P', bP)]:
    view[c + '_x'] = (view[c] / b).round(2)

print('=' * 88)
print(' [버전 A] 라이벌전 vs 평상시  —  M_t = M_0 * exp(log_shock)')
print('=' * 88)
print(view[['game_id', 'M0', 'shock_x', 'M_t', 'capacity',
            'p_D7_G_x', 'p_D7_P_x', 'p_D3_G_x', 'p_D3_P_x',
            'audience', 'revenue', 'corner_hits']].to_string(index=False))

rev_rival  = res.loc[res['game_id'].str.contains('Rams'),    'revenue'].values[0]
rev_normal = res.loc[res['game_id'].str.contains('Falcons'), 'revenue'].values[0]

print()
print('-' * 88)
print(f"  평상시  (Jaguars vs Falcons) 기대수익 : {rev_normal:>15,} 원   (지수 100.0)")
print(f"  라이벌전 (Jaguars vs Rams)    기대수익 : {rev_rival:>15,} 원   "
      f"(지수 {rev_rival / rev_normal * 100:.1f})")
print(f"  수익 증가율 : {(rev_rival / rev_normal - 1) * 100:+.1f}%")
print('-' * 88)

 [버전 A] 라이벌전 vs 평상시  —  M_t = M_0 * exp(log_shock)
           game_id    M0  shock_x   M_t  capacity  p_D7_G_x  p_D7_P_x  p_D3_G_x  p_D3_P_x  audience   revenue  corner_hits
   Jaguars vs Rams 17575    1.419 24940     23750      1.25      1.20      1.39      1.27     15523 346332709            0
Jaguars vs Falcons 17575    1.000 17575     23750      1.33      1.11      1.41      1.18     10912 265107695            0

----------------------------------------------------------------------------------------
  평상시  (Jaguars vs Falcons) 기대수익 :     265,107,695 원   (지수 100.0)
  라이벌전 (Jaguars vs Rams)    기대수익 :     346,332,709 원   (지수 130.6)
  수익 증가율 : +30.6%
----------------------------------------------------------------------------------------


## [Example] 추가 분석 진행

In [41]:
# ==============================================================================
# 10. [진단] D-7 가격 역전 현상 해부
#      "수요가 늘었는데 왜 D-7 가격은 오히려 내려가는가?"
# ==============================================================================
print('=' * 88)
print(' [진단] 시점별 내부 상태 비교')
print('=' * 88)

for _, r in res.iterrows():
    px = [r['p_D7_G'], r['p_D7_P'], r['p_D3_G'], r['p_D3_P']]
    d = revenue_detail(px, r['M_t'], r['capacity'])

    print(f"\n▶ {r['game_id']}   (M_t = {r['M_t']:,} / capacity = {r['capacity']:,})")
    print(f"   시장배분   : M_D7 = {d['M_D7']:>9,.0f}   M_D3 = {d['M_D3']:>9,.0f}")
    print(f"   좌석용량   : cap_G = {d['cap_G']:>8,.0f}   cap_P = {d['cap_P']:>8,.0f}")
    print(f"   D-7 가격   : G {r['p_D7_G']:>7,} ({r['p_D7_G']/bG:.2f}x)   "
          f"P {r['p_D7_P']:>7,} ({r['p_D7_P']/bP:.2f}x)")
    print(f"   D-7 판매   : G {d['Q_D7_G']:>8,.0f}          P {d['Q_D7_P']:>8,.0f}")
    print(f"   ─ 잔여재고율 r_G = {d['r_G']:.3f}  →  희소성 f_G = {d['f_G']:.3f}  "
          f"→  유효민감도 γ_e,G = {d['gamma_eff_G_D3']:.6f}")
    print(f"   D-3 가격   : G {r['p_D3_G']:>7,} ({r['p_D3_G']/bG:.2f}x)   "
          f"P {r['p_D3_P']:>7,} ({r['p_D3_P']/bP:.2f}x)")
    print(f"   D-3 판매   : G {d['Q_D3_G']:>8,.0f}          P {d['Q_D3_P']:>8,.0f}")
    print(f"   총 관중    : {d['total_audience']:>8,.0f}      수익 : {d['revenue']:>15,.0f} 원")

print('\n' + '─' * 88)
print(' 해석: 라이벌전은 D-3 시장(M_D3)이 훨씬 크다. 따라서 최적해는 D-7 가격을')
print('       상대적으로 낮춰 재고를 미리 소진시키고(잔여재고율↓ → 희소성 f↑ →')
print('       유효민감도 γ_e↓), 그 결과 가격 저항이 약해진 D-3에서 더 많은 물량을')
print('       더 비싸게 파는 전략을 택한다. "수요가 늘면 무조건 올린다"는 직관과 다르다.')
print()
print(' [중요] 위 가격은 모두 0.7x~1.5x 경계에 붙지 않은 "내부해"이다.')
print('        만약 어떤 가격이 1.50x 에 딱 붙어 있다면, 그것은 최적해가 아니라')
print('        "더 올리고 싶은데 상한에 막힌" 코너해이며, 시나리오 간 비교가 왜곡된다.')
print('─' * 88)

 [진단] 시점별 내부 상태 비교

▶ Jaguars vs Rams   (M_t = 24,940 / capacity = 23,750)
   시장배분   : M_D7 =     9,976   M_D3 =    14,964
   좌석용량   : cap_G =   20,188   cap_P =    3,562
   D-7 가격   : G  15,012 (1.25x)   P  41,976 (1.20x)
   D-7 판매   : G    4,892          P    1,451
   ─ 잔여재고율 r_G = 0.758  →  희소성 f_G = 1.073  →  유효민감도 γ_e,G = 0.000168
   D-3 가격   : G  16,684 (1.39x)   P  44,543 (1.27x)
   D-3 판매   : G    7,068          P    2,111
   총 관중    :   15,523      수익 :     346,330,581 원

▶ Jaguars vs Falcons   (M_t = 17,575 / capacity = 23,750)
   시장배분   : M_D7 =     7,030   M_D3 =    10,545
   좌석용량   : cap_G =   20,188   cap_P =    3,562
   D-7 가격   : G  15,943 (1.33x)   P  38,856 (1.11x)
   D-7 판매   : G    2,963          P    1,465
   ─ 잔여재고율 r_G = 0.853  →  희소성 f_G = 1.044  →  유효민감도 γ_e,G = 0.000172
   D-3 가격   : G  16,954 (1.41x)   P  41,274 (1.18x)
   D-3 판매   : G    4,386          P    2,097
   총 관중    :   10,912      수익 :     265,107,463 원

─────────────────────────────────────────────

In [42]:
# ==============================================================================
# 11. [부록] 균일 가격배수 스윕 — 최적화 없이 단순 인상하면?
# ==============================================================================
row_r = df_games_demo.iloc[0]
M0_r  = M0_df.query("team=='Jaguars' and season==2024")['M0'].values[0]
cap_r = CAPACITY_BASE['Jaguars']
M_t_r = M0_r * np.exp(my_virtual_indicators(row_r))

print(f"라이벌전 (M_t = {M_t_r:,.0f}, capacity = {cap_r:,})")
print(f"{'배수':>6} | {'수익(원)':>16} | {'관중':>9} | {'지수':>7}")
print('-' * 50)

base_rev = revenue_detail([bG, bP, bG, bP], M_t_r, cap_r)['revenue']
for m in [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    d = revenue_detail([bG * m, bP * m, bG * m, bP * m], M_t_r, cap_r)
    print(f"{m:6.1f} | {d['revenue']:16,.0f} | {d['total_audience']:9,.0f} | "
          f"{100 * d['revenue'] / base_rev:7.1f}")

opt = optimize_single_game(M_t_r, cap_r)
d_opt = revenue_detail(opt.x, M_t_r, cap_r)
print('-' * 50)
print(f"{'최적화':>6} | {d_opt['revenue']:16,.0f} | {d_opt['total_audience']:9,.0f} | "
      f"{100 * d_opt['revenue'] / base_rev:7.1f}")
print('\n→ 균일 인상으로는 최적해에 도달할 수 없다. 시점·등급별 차등이 핵심이다.')

라이벌전 (M_t = 24,940, capacity = 23,750)
    배수 |            수익(원) |        관중 |      지수
--------------------------------------------------
   0.7 |      199,043,265 |    16,868 |    67.8
   0.8 |      231,797,576 |    17,317 |    79.0
   0.9 |      263,594,005 |    17,579 |    89.9
   1.0 |      293,366,810 |    17,619 |   100.0
   1.1 |      320,076,451 |    17,420 |   109.1
   1.2 |      341,298,418 |    16,944 |   116.3
   1.3 |      325,612,568 |    15,537 |   111.0
   1.4 |      305,857,329 |    14,061 |   104.3
   1.5 |      283,023,582 |    12,553 |    96.5
--------------------------------------------------
   최적화 |      346,332,709 |    15,523 |   118.1

→ 균일 인상으로는 최적해에 도달할 수 없다. 시점·등급별 차등이 핵심이다.


# [참가자 작성 영역] 가상지표 정의

### 수식 설명
$$\log(\text{shock}_t) = \sum_j \delta_j \, V_{j,t} \qquad\Longrightarrow\qquad M_t = M_0 \cdot e^{\log(\text{shock}_t)}$$

| 기호 | 의미 | 부호 해석 |
|---|---|---|
| $V_{j,t}$ | 가상지표 $j$의 발동 여부(0/1) 또는 강도 | — |
| $\delta_j$ | 지표 $j$의 **로그 스케일 충격 계수** | $\delta>0$ 수요↑ / $\delta<0$ 수요↓ |
| $e^{\delta}$ | 실제 **수요 배수** | $\delta=0.35 \Rightarrow e^{0.35}\approx1.42$ (약 42% 증가) |

In [43]:
# ==============================================================================
# [P-1] [전 시나리오 공통] 월·요일 달력 지표 (일괄적용)
# ==============================================================================
#  참가자 작성 영역 구성
#   · [P-1] 공통 달력 / [P-2] 대기록 / [P-3] 부상 / [P-4] 강수 / [P-5] 통합·쌍체 비교
#   · [A-1]~[A-2] 공통 검증 / [A-3] 대기록 / [A-4]~[A-10] 부상 / [A-11] 강수 분석
#   · delta_calendar 는 전 경기에 일괄 적용하고, 세 시나리오 지표는 각 이벤트 정보에
#     따라 선별 적용한다. 네 지표는 모두 구현 완료 상태다.
#   · 달력 지표 때문에 조직위 대조군도 log_shock != 0 이다. 조직위 대조군/처치군은
#     정책 총효과로만 읽고, 이벤트 순효과는 [P-5]의 동일경기 쌍체 반사실로 산출한다.
#
#  식(0-1)  delta_month(t)  = ln( 월 상대비율 )
#  식(0-2)  delta_dow(t)    = ln( 요일 상대비율 )
#  식(0-3)  delta_cal(t)    = delta_month + delta_dow          <- 플래그 없이 전 경기 적용
#
#  근거: KBO 공식 기록 2023·2024 시즌 9/15 기준 집계자료
#        표3 월별 경기수·총관중 / 표4 요일별 평균관중. 시즌 종료 전체 자료가 아니라
#        두 시즌을 동일한 기준일로 맞춘 공식 집계 스냅샷으로 사용한다.
#        원자료와 산출 과정은 [A-1] 에서 그대로 재현한다(하드코딩한 비율이 아니다).
#
#  산출 2단계
#    1단계  각 해 '리그 전체 평균 관중' 대비 상대비율을 구한다.
#           (연도 절대상승 = 리그 인기 추세를 제거. 그 추세는 M0 가 이미 반영한다)
#           2024 계 671경기 14,934명 / 2023 계 660경기 11,033명(월별)·11,233명(요일별)
#           월 비율은 시즌평균이 월평균의 경기가중 평균이므로 구성상 정확히 중심화된다.
#    2단계  2개년 평균비율을 구간 집약 후 log 변환한다. delta = ln(비율)
#
#  월 (3구간)                              요일 (2구간, KBO 는 화~일 6일 편성)
#    성수기 3,9월   1.064 -> ln +0.062       주말 금·토·일  1.194 -> ln +0.177
#    평수기 5~8월   1.007 -> ln +0.007       주중 화·수·목  0.794 -> ln -0.231
#    비수기 4월     0.922 -> ln -0.081       ※ 월요일 편성분은 주중으로 처리한다
#                                              (KBO 는 월요일 경기가 없어 화수목으로 추정)
#
#  평수기 ln 은 +0.007 로 사실상 0이다. 반올림 결과가 아니라 '평수기를 시즌 평균
#  기준선으로 둔다'는 설계 선택이며, 그래서 0.00 을 부여한다.
#  읽히는 패턴: 3·9월(시즌 시작·끝)이 높고 4월이 유일한 저점. 7월은 장마와 여름방학이
#  상쇄돼 평년 수준이다. 요일 격차(+-0.20)가 월 격차(+-0.06~0.08)보다 3배 크다.
#
#  [표본 주의] 2023 시즌은 4월 개막이라 3월 자료가 없다. 성수기 구간의 3월은 2024 단독
#  37경기, 9월은 2개년 160경기다. 성수기 +0.06 은 사실상 9월이 주도한다.
#
#  [주의] 일괄적용이므로 각 시나리오의 대조군도 delta != 0 이 된다. 대조군을 반사실로
#  쓸 수 없고, 이벤트 순효과는 반드시 동일경기 쌍체 반사실로 산출한다. ([P-5], [A-4])
# ==============================================================================
#  아래 상수는 관중 증감률을 직접 더하는 값이 아니라 수요에 곱해질 로그충격이다.
#  예: +0.18은 정확히 18% 증가가 아니라 exp(0.18)=1.197배의 수요배수를 뜻한다.
CAL_MONTH = {3: 0.06, 9: 0.06, 4: -0.08}      # key가 없는 5~8월은 기준선 0.00
CAL_WEEKEND_DAYS = {'금', '토', '일'}          # KBO 흥행 패턴에 맞춘 주말 묶음
CAL_WEEKEND, CAL_WEEKDAY = 0.18, -0.23         # 월요일은 else 분기에서 주중으로 처리


def delta_calendar(row):
    """[공통] 월·요일 로그충격. 일괄적용. 식(0-1)+(0-2) = 식(0-3)"""
    month = int(str(row['date'])[5:7])                       # '2024-09-02' -> 9
    d = CAL_MONTH.get(month, 0.0)                            # 식(0-1): 미등록 월은 평수기 0
    d += CAL_WEEKEND if row['day_of_week'] in CAL_WEEKEND_DAYS else CAL_WEEKDAY   # 식(0-2)
    return d                                                 # 식(0-3), exp(d)가 달력 수요배수

In [44]:
# ==============================================================================
# [P-2] [시나리오 1] 스타 대기록 — 체이스 화제성 + 해당 경기 달성확률
# ==============================================================================
#  식(1-1)  lambda_HR = (시즌 기준 홈런 수 / 144경기) x 파크팩터
#  식(1-2)  P_moment(h) = P(N >= h),  N ~ Poisson(lambda_HR)
#  식(1-3)  delta_star = is_star_record x (D_CHASE + D_MOMENT x P_moment)
#
#  해석
#   · D_CHASE 는 대기록 도전 구간 전체에 걸리는 공통 화제성이다.
#   · D_MOMENT 는 해당 경기에서 기록 달성을 직접 목격할 확률에 비례하는 추가 효과다.
#   · 처치군은 3->2->1->0 순으로 달성확률이 비선형 상승한다.
#   · hr_from_record 는 사후 관측치가 아니라 가격 결정 시점에 이미 공표돼 있는 전망치다.
#     조직위 FAQ Q18은 "기록까지 3개 남은 상태에서 선수 페이스가 올라 1주 뒤 홈 4연전
#     중 달성이 유력"하다는 전제를 출제진이 모형화해 3->2->1->0 경로를 고정 제공한
#     것이라고 명시했고, 이 점에서 rain_prob 과 같은 성격이며 당일 실현치인
#     rain_at_game_time 과는 구분된다고 답했다. 따라서 본 모형은 이 값을 D-7/D-3
#     가격 결정 시점의 사전 정보로 사용한다.
#   · P_moment(0)=1 은 포아송 계산 결과가 아니라 위 시나리오 전제다. 조직위 정의상
#     hr_from_record=0 은 "그 경기에서 달성"을 뜻하므로 확률이 아니라 전제로서 1이 된다.
#     그 결과 delta_star 는 hr=1 의 0.227 에서 hr=0 의 0.450 으로 한 번 불연속 상승하는데,
#     이는 모형의 결함이 아니라 고정된 전제 경로를 그대로 옮긴 것이다. 보고서에도
#     "달성확률 100%"가 아니라 "시나리오 전제상 달성 경기"로 서술한다.
#   · 운영 한계. FAQ Q18 이 밝히듯 실제 운영에서는 매 경기 결과에 따라 잔여 홈런 수를
#     갱신해야 한다. 본 과제는 시나리오별 가격 대응 전략을 비교하기 위해 경로를 고정했으므로,
#     실무 적용 시에는 D-7 과 D-3 사이에 hr_from_record 가 갱신되면 재가격을 트리거하는
#     규칙이 별도로 필요하다.
#   · 달력 편중. 대조군 4월 / 처치군 6월로 delta_cal 격차 +0.080 (수요배수 1.083) 이
#     처치군에 유리하다. 순효과는 반드시 쌍체 반사실로 산출한다.
# ==============================================================================
from scipy.stats import poisson

STAR = {
    'hr_count':   42,    # 홈런왕 평균 41.8개의 근사값
    # 실제 선수 출전 경기 수가 아니라, 144경기 시즌 기준 경기당 홈런율 환산용 분모다.
    # 선수의 144경기 전 경기 출전을 뜻하지 않는다.
    'games':      144,
    'park_factor': 1.02, # 기준안 1.02 / 민감도 0.74, 1.47
    'd_chase':    0.15,  # 공통 화제성 로그충격 / 민감도 0.10, 0.20
    'd_moment':   0.30,  # 달성 목격확률 추가 로그충격 상한 / 민감도 0.20, 0.40
}

# 보고서 표기와 기존 분석 셀의 가독성을 위한 기준안 별칭
PLAYER_HR_COUNT = STAR['hr_count']
PLAYER_GAMES = STAR['games']
PARK_FACTOR = STAR['park_factor']
D_CHASE = STAR['d_chase']
D_MOMENT = STAR['d_moment']
LAMBDA_HR = PLAYER_HR_COUNT / PLAYER_GAMES * PARK_FACTOR       # 식(1-1)


def _scenario1_scalar(row, key, default=None):
    """시나리오 1 입력을 scalar 로 읽고 None/NaN 을 동일하게 처리한다."""
    value = row.get(key, default)
    if value is None or pd.isna(value):
        return default
    return value


def _p_moment(hr, lam=LAMBDA_HR):
    """그 경기에서 기록을 달성할 확률 P(N>=hr), N~Poisson(lam). 식(1-2)"""
    try:
        hr = float(hr)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'hr_from_record는 0 이상의 정수여야 합니다: {hr!r}') from exc
    if not np.isfinite(hr) or hr < 0 or not hr.is_integer():
        raise ValueError(f'hr_from_record는 0 이상의 정수여야 합니다: {hr!r}')
    if hr == 0:
        return 1.0          # 조직위 정의: 해당 경기에서 기록 달성
    return float(poisson.sf(int(hr) - 1, lam))


def delta_star_record(row, p=STAR):
    """[시나리오 1] 체이스 화제성 + 해당 경기 달성확률의 로그충격. 식(1-3)"""
    flag = _scenario1_scalar(row, 'is_star_record', 0)
    if flag not in (0, 1, False, True):
        raise ValueError(f'is_star_record는 0 또는 1이어야 합니다: {flag!r}')
    if int(flag) == 0:
        return 0.0

    hr = _scenario1_scalar(row, 'hr_from_record')
    if hr is None:
        raise ValueError('is_star_record=1인 행에는 hr_from_record가 필요합니다.')
    lam = p['hr_count'] / p['games'] * p['park_factor']
    return p['d_chase'] + p['d_moment'] * _p_moment(hr, lam)

In [45]:
# ==============================================================================
# [P-3] [시나리오 2] 핵심 타자 부상 — WAR 전력손실 + 지속하한형 시간감쇠
# ==============================================================================
#  [모형을 이렇게 구성한 이유]
#  타율·홈런·수비포지션·결승타를 별도로 더하면 동일 선수의 경기력을 여러 번 계산할 수
#  있다. WAR는 타격·주루·수비·포지션 조정을 하나의 "대체선수 대비 추가 승수"로 합친
#  지표이므로, 핵심 타자를 대체선수로 바꾸는 부상 상황과 가장 직접적으로 대응한다.
#
#  [기호와 단위]
#   · WAR_gap = starter_war - backup_war : 144경기 시즌 기준 전력 차이[승]
#   · G_miss  = games_miss                : 62일 결장구간의 설명용 예상 경기 수[경기]
#   · d       = days_since_injury         : 부상 발생 후 경과한 달력 일수[일]
#   · W       = win_rate                  : 부상 전 경기당 기대승수[승/경기]
#   · beta    = attendance_beta           : 절대 승률 1.0 변화당 로그관중 변화 계수
#   · rho     = persistent_floor          : 장기 지속효과 하한(0~1)
#   · h       = attention_half_life_days  : 감쇠되는 초기 관심 부분의 반감기[일]
#
#  [사용 식]
#  식(2-1) WAR_lost = WAR_gap × G_miss / season_games
#                     전체 시즌 WAR 격차 중 결장 구간에 해당하는 예상 승수 손실이다.
#  식(2-2) Delta_W  = -WAR_lost/G_miss = -WAR_gap/season_games
#                     경기당 절대 승률 하락폭이다. G_miss는 정확히 약분되므로
#                     games_miss는 WAR_lost를 설명할 때만 사용되고 최종 delta에는 무관하다.
#  식(2-3) A(d)     = rho + (1-rho)×2^(-d/h),  0 <= d < days_miss
#                     d=0에는 A=1이고, 시간이 지나면 rho에 수렴한다. 순수 지수감쇠처럼
#                     58일째 효과가 거의 0이 되지 않도록 실제 결장 중인 지속효과를 남긴다.
#  식(2-4) d_str    = beta × Delta_W × A(d)
#                     KBO 패널 추정식 Delta ln(관중)=1.241×Delta 승률을 옮긴다.
#  식(2-5) d_pop    = 0
#                     경기력 경로는 식(2-1)~(2-4)의 WAR→승률→관중으로 이미 반영한다.
#                     Park(2023)의 시즌 단위 해외이적 연구를 추가 인기효과가 뚜렷하지
#                     않다는 간접 근거로 사용하고, 중복·자의적 가산을 피하기 위해 별도의
#                     순수 인기충격은 보수적으로 0을 둔다. 이는 단기 부상효과의 직접
#                     추정값이나 '인기효과가 정확히 0'이라는 결론이 아니다.
#  식(2-6) delta    = is_key_injury × 1[0 <= d < days_miss] × (d_str+d_pop)
#                     고정 파이프라인에 넣는 로그충격이다. 실제 수요배수는 exp(delta)다.
#
#  [A(d)의 정확한 해석과 한계]
#   · A(d)는 선수의 실력이 시간이 지나며 사라진다는 뜻이 아니다. 결장 직후의 뉴스성·
#     검색 관심·소비자 충격은 빠르게 약해지지만, 선수가 계속 결장하면 라인업 전력손실과
#     경기 결과 불확실성 변화는 남는다는 두 현상을 하나의 축약 가중치로 표현한다.
#   · rho=0.75를 직접 추정한 KBO 논문은 찾지 못했다. 따라서 0.75는 "초기효과의 75%를
#     결장 지속효과로 남긴다"는 구조적 기준 가정이지 실증 추정치가 아니다. 이 불확실성은
#     [A-7]에서 rho={0.50,0.75,1.00} 전체를 공개하고 결과 부호를 검증한다.
#   · h=8은 스포츠 티켓수요의 직접 추정치가 아니다. 아래 일반 미디어 연구가 사건 보도의
#     지수감쇠와 8일 반감기를 직접 추정했고, NBA·NFL 부상 검색 연구의 3~6일 집중구간과
#     방향이 정합적이어서 "초기 관심 부분"의 대리 기준으로만 채택한다.
#   · 기존 14일은 직접 근거가 없어 기준값에서 제외하되 [A-7]의 느린 감쇠 스트레스로 남긴다.
#   · rho=0.75이면 h=8과 h=14의 차이는 작고 rho 선택이 더 중요하다. 2·14·30·45·58일의
#     A(d)는 h=8에서 0.9602·0.8243·0.7686·0.7551·0.7516이다.
#
#  [기준값의 출처와 산출 — URL/논문명을 주석에 남겨 재검색 가능하게 함]
#   1) season_games=144
#      KBO, "2024 KBO 정규시즌 경기 일정 확정", 팀당 144경기.
#      https://www.koreabaseball.com/MediaNews/Notice/View.aspx?bdSe=9926
#   2) regular_season_days=193
#      위 KBO 공지의 2024-03-23 개막일부터 KBO의 2024-10-01 정규시즌 종료일까지
#      양끝을 포함해 계산한 달력일이다.
#      https://www.koreabaseball.com/MediaNews/Notice/View.aspx?bdSe=10231
#   3) days_miss=62
#      조직위 고정 시나리오의 2024-07-01 부상 후 2개월(7/1~8/31)을 양끝 포함한 값이다.
#      외부 통계 추정치가 아니라 주어진 시나리오 조건이다.
#   4) games_miss=144×62/193=46.26
#      시즌 평균 진행률을 적용한 설명용 기대 경기 수다. 실제 팀별 우천순연 편성을
#      재현한 값은 아니며 식(2-2)에서 약분되므로 최종 수요충격 크기에는 영향을 주지 않는다.
#   5) starter_war=5.0, backup_war=0.0
#      FanGraphs Library, "What Is WAR?"의 5~6 WAR=Superstar 구간 하한과
#      "Replacement Level"의 0 WAR 기준을 사용한다.
#      https://library.fangraphs.com/misc/war/
#      https://library.fangraphs.com/misc/war/replacement-level/
#   6) win_rate=78/(78+64)=0.5493
#      가상자료 home_rank=2를 2024 KBO 2위 삼성(78승 64패 2무)에 대응시킨 검산값이다.
#      KBO 승률 계산처럼 무승부는 분모에서 제외한다. 최종 delta에는 곱하지 않고
#      부상 후 기대승률이 현실적인 범위인지 확인할 때만 사용한다.
#      https://www.koreabaseball.com/MediaNews/News/KboPhoto/View.aspx?bdSe=465022
#   7) attendance_beta=1.241
#      Eun Cheon Jeong & Giseok Nam (2021),
#      "The Habitual Consumption Propensity of Korean Professional Baseball Fans
#       for Sustainable Management", International Journal of Entrepreneurship 25(2S).
#      KBO 1982~2014 팀-시즌 패널에서 승률 변화 계수 1.241(t=6.04, 1% 유의)를 보고한다.
#      https://www.abacademies.org/articles/the-habitual-consumption-propensity-of-korean-professional-baseball-fans-for-sustainable-management-12628.html
#      주의: 팀-시즌 관계이지 단일 경기 부상 공지의 인과계수는 아니다. 가상지표 실측값이
#      없다는 요강 조건에서 국내 KBO 실증치를 기준점으로 이전한다.
#   8) d_pop=0
#      Jeong Seung Park (2023), "Identifying the Popularity Effect of a Star Player
#      on Sports Attendance Demand: Evidence from the Korea Baseball Organization Data",
#      고려대학교 석사학위논문, DOI 10.23186/korea.000000276603.11009.0000125.
#      Park은 이승엽의 KBO 활동기(1995~2003, 2012~2017)와 해외 활동기(2004~2011)를
#      중심으로 한 시즌 단위 Bayesian synthetic control에서 팀 경기력을 통제한 뒤에도
#      해외이적이 홈 관중의 유의한 감소를 만들었는지 검토했다. 결과가 유의하지 않았다는
#      점은 '경기력 외의 추가 인기효과가 크다'고 단정하지 않는 간접 근거로만 사용한다.
#      이 연구는 여러 시즌의 해외이적을 다루므로 62일 부상이나 발표 직후 개별 경기의
#      인기효과를 직접 추정하지 않았고, 유의하지 않다는 결과도 효과가 정확히 0이라는
#      증명은 아니다. 본 모형에서는 경기력 손실이 이미 d_str의 WAR→승률→로그관중 경로에
#      반영되어 있고, 별도의 KBO 단기 부상 인기계수가 없으므로 중복계산과 자의적 가산을
#      피하는 보수적 기준안으로 추가 순수 인기항 d_pop=0을 설정한다.
#      https://dcollection.korea.ac.kr/srch/srchDetail/000000276603
#
#  [부상 효과의 부호·지속성·시간감쇠를 판단할 때 함께 검토한 보조 연구]
#   9) Patrick J. Ferguson & Karim R. Lakhani (2021),
#      "Consuming Contests: Outcome Uncertainty and Spectator Demand for Contest-based
#       Entertainment", HBS Working Paper 21-087.
#      AFL에서 현재 부상 라인업 변경 1표준편차 증가와 관중 2.1~4.9% 감소가 연결되고,
#      현재 부상만 현재 경기 관중과 유의한 관계를 보였다. 이는 부상효과의 음(-)의 부호와
#      실제 결장 중 지속효과 하한의 필요성을 보조하지만 KBO 계수로 직접 사용하지 않는다.
#      https://www.hbs.edu/ris/Publication%20Files/21-087_007ff15a-97d8-481a-aff0-d63f0c300b7e.pdf
#  10) Scott M. Kaplan (2022), "Putting a Price on Popularity: Evidence from
#      Superstars in the National Basketball Association", Economic Inquiry 60(2),
#      DOI 10.1111/ecin.13065.
#      스타 결장 발표 후 2차시장 가격 4~22% 하락을 추정하지만 경기 3일 이내 발표 표본이며,
#      장기 결장 발표의 시간경로는 후속 연구과제로 남긴다. 따라서 부호와 단기 관련성의
#      보조 근거일 뿐 rho나 h의 직접 추정치가 아니다.
#      https://doi.org/10.1111/ecin.13065
#  11) Patrick Reilly, John L. Solow & Peter von Allmen (2023),
#      "When the Stars Are Out: The Impact of Missed Games on NBA Television Audiences",
#      Journal of Sports Economics 24(7), DOI 10.1177/15270025231174616.
#      스타 결장이 TV 시청과 광고가치에 음의 영향을 준다는 보조 근거이며 시간감쇠는 추정하지 않는다.
#      https://doi.org/10.1177/15270025231174616
#  12) Brad R. Humphreys & Candon Johnson (2020),
#      "The Effect of Superstars on Game Attendance: Evidence From the NBA",
#      Journal of Sports Economics 21(2), DOI 10.1177/1527002519885441.
#      일부 스타의 홈·원정 경기 출전이 관중 증가와 연결됨을 보여주는 방향성 근거다.
#      https://doi.org/10.1177/1527002519885441
#  13) Jay C. Thompson et al. (2021),
#      "An Analysis of Google Trends Following Athletic Injuries by High Profile NBA
#       Players During the 2019 NBA Finals", Journal of Osteopathic Medicine 121(6),
#      DOI 10.1515/jom-2020-0269.
#      부상 관련 검색 관심의 유의한 증가가 평균 6.13일(SD 3.14일) 이어졌다고 보고한다.
#      https://doi.org/10.1515/jom-2020-0269
#  14) Trevor Torgerson et al. (2020),
#      "Public Online Interest in Injuries Sustained by National Football League
#       Quarterbacks", Arthroscopy, Sports Medicine, and Rehabilitation 2(5),
#      DOI 10.1016/j.asmr.2020.07.009.
#      NFL 쿼터백 부상 후 첫 3일을 핵심 온라인 관심구간으로 확인한다.
#      https://doi.org/10.1016/j.asmr.2020.07.009
#  15) Rafael Prieto Curiel et al. (2019),
#      "Temporal and Spatial Analysis of the Media Spotlight",
#      Computers, Environment and Urban Systems 75, 254-263,
#      DOI 10.1016/j.compenvurbsys.2019.02.004.
#      사건 보도가 지수적으로 감소하고 8일마다 절반이 된다고 직접 추정한다. 스포츠 수요
#      연구가 아니므로 h=8은 "뉴스 관심 부분"의 대리값이며 민감도 없이 사실로 단정하지 않는다.
#      https://doi.org/10.1016/j.compenvurbsys.2019.02.004
#
#  [결과 해석 시 반드시 지킬 점]
#   · 조직위 대조군과 처치군은 월·요일 구성이 다르므로 단순 평균 차이를 부상효과로 읽지
#     않는다. [P-5]/[A-4]에서 같은 처치경기의 플래그만 끄는 쌍체 반사실을 사용한다.
#   · 고정 파이프라인의 SHOCK_CLIP·MNL 수요·좌석용량·SLSQP는 수정하지 않는다.
#   · d=62가 되는 순간 delta=0으로 바뀌는 확정 복귀 게이트이므로 실제 복귀일 불확실성과
#     단계적 복귀는 표현하지 못한다.
# ==============================================================================
INJURY = {
    'starter_war':          5.0,             # FanGraphs Superstar 5~6 WAR의 하한
    'backup_war':           0.0,             # FanGraphs replacement level 기준
    'regular_season_days': 193,              # 2024-03-23~10-01 양끝 포함
    'days_miss':            62,               # 조직위 2개월: 2024-07-01~08-31
    'season_games':         144,              # KBO 공식 팀당 정규시즌 경기 수
    'games_miss':           144 * 62 / 193,   # 설명용 46.26경기; 식(2-2)에서 약분
    'win_rate':             78 / (78 + 64),   # 2024 KBO 2위 삼성; 무승부 제외
    'attendance_beta':      1.241,            # KBO Delta ln(관중)/Delta 승률
    'persistent_floor':     0.75,             # 직접 추정치가 아닌 구조적 기준 가정
    'attention_half_life_days': 8.0,          # 일반 미디어의 8일 반감기 대리값
}


def _injury_attenuation(days_since_injury, p=INJURY):
    """식(2-3)의 지속하한형 시간가중치 A(d)를 계산한다."""
    d = float(days_since_injury)
    # A(d)는 수식상 d>=0에서만 정의한다. 음수는 호출부에서 '부상 전=효과 0'으로
    # 걸러야 하며, 여기서 d=0으로 바꾸면 식(2-6)의 1[0<=d]와 불일치한다.
    if not np.isfinite(d) or d < 0.0:
        raise ValueError(f'days_since_injury는 0 이상의 유한수여야 합니다: {d!r}')
    rho = float(p['persistent_floor'])
    h = float(p['attention_half_life_days'])
    if not 0.0 <= rho <= 1.0:
        raise ValueError(f'persistent_floor는 0~1이어야 합니다: {rho!r}')
    if not np.isfinite(h) or h <= 0.0:
        raise ValueError(f'attention_half_life_days는 양수여야 합니다: {h!r}')
    return rho + (1.0 - rho) * 2.0 ** (-d / h)                 # 식(2-3)


def delta_injury(row, p=INJURY, detail=False):
    """[시나리오 2] 한 경기의 핵심 타자 부상 로그충격을 계산한다.

    detail=False는 고정 파이프라인에 넣을 최종 delta 하나를 반환한다.
    detail=True는 보고서 검산용 식(2-1)~(2-6)의 모든 중간값을 반환한다.
    """
    zero = dict(war_lost=0.0, delta_win_rate=0.0, win_rate_after=p['win_rate'],
                A=0.0, d_str=0.0, d_pop=0.0, delta=0.0)

    d = row.get('days_since_injury')
    flag = row.get('is_key_injury', 0)
    if pd.isna(flag):
        flag = 0
    if int(flag) != 1 or d is None or pd.isna(d):
        return zero if detail else 0.0
    d = float(d)
    # 식(2-6)의 1[0<=d<days_miss]를 그대로 구현한다. d<0은 부상 발생 전이므로
    # 최대 충격인 d=0으로 보정하지 않고 비활성(로그충격 0)으로 처리한다.
    if not np.isfinite(d) or d < 0.0 or d >= p['days_miss']:
        return zero if detail else 0.0

    war_gap = p['starter_war'] - p['backup_war']
    war_lost = war_gap * p['games_miss'] / p['season_games']   # 식(2-1)
    delta_win_rate = -war_lost / p['games_miss']               # 식(2-2)
    win_rate_after = p['win_rate'] + delta_win_rate
    A = _injury_attenuation(d, p)                              # 식(2-3)
    d_str = p['attendance_beta'] * delta_win_rate * A           # 식(2-4)
    # 식(2-5): 경기력 효과는 d_str에 이미 포함되어 있다. Park(2023)은 시즌 단위의
    # 간접 근거이며 단기 부상 인기효과의 직접 추정치는 아니므로, 중복·자의적 가산을
    # 피하는 보수적 기준안으로 별도 추가항만 0으로 둔다.
    d_pop = 0.0
    out = dict(war_lost=war_lost, delta_win_rate=delta_win_rate,
               win_rate_after=win_rate_after, A=A, d_str=d_str,
               d_pop=d_pop, delta=d_str + d_pop)                # 식(2-6)
    return out if detail else out['delta']


_gap = INJURY['starter_war'] - INJURY['backup_war']
_d0 = delta_injury({'is_key_injury': 1, 'days_since_injury': 0}, detail=True)
print(f"[시나리오 2] WAR_gap {_gap} = 주전 {INJURY['starter_war']} - 백업 {INJURY['backup_war']}")
print(f"  식(2-1) WAR_lost = {_gap} × {INJURY['games_miss']:.4f}/{INJURY['season_games']}"
      f" = {_d0['war_lost']:.4f}승")
print(f"  식(2-2) Delta_W = -{_gap}/{INJURY['season_games']}"
      f" = {_d0['delta_win_rate']:+.6f}; 부상 후 검산승률 {_d0['win_rate_after']:.4f}")
print(f"  식(2-3) A(d) = {INJURY['persistent_floor']}"
      f" + (1-{INJURY['persistent_floor']})×2^(-d/{INJURY['attention_half_life_days']})")
for _day in [2, 14, 30, 45, 58]:
    _det = delta_injury({'is_key_injury': 1, 'days_since_injury': _day}, detail=True)
    print(f"    d={_day:>2}  A={_det['A']:.4f}  delta={_det['delta']:+.5f}"
          f"  수요변화={(np.exp(_det['delta'])-1)*100:+.2f}%")
print("  ※ rho=0.75는 직접 추정치가 아닌 구조적 가정이며 [A-7]의 rho×반감기 격자로 검증한다.")


[시나리오 2] WAR_gap 5.0 = 주전 5.0 - 백업 0.0
  식(2-1) WAR_lost = 5.0 × 46.2591/144 = 1.6062승
  식(2-2) Delta_W = -5.0/144 = -0.034722; 부상 후 검산승률 0.5146
  식(2-3) A(d) = 0.75 + (1-0.75)×2^(-d/8.0)
    d= 2  A=0.9602  delta=-0.04138  수요변화=-4.05%
    d=14  A=0.8243  delta=-0.03552  수요변화=-3.49%
    d=30  A=0.7686  delta=-0.03312  수요변화=-3.26%
    d=45  A=0.7551  delta=-0.03254  수요변화=-3.20%
    d=58  A=0.7516  delta=-0.03239  수요변화=-3.19%
  ※ rho=0.75는 직접 추정치가 아닌 구조적 가정이며 [A-7]의 rho×반감기 격자로 검증한다.


In [46]:
# ==============================================================================
# [P-4] [시나리오 3] 기습 장마 예보 — 강수확률 로지스틱 충격
# ==============================================================================
#  식(3-1)  delta_rain(q) = -K / {1 + exp[-a(q-q0)]}
#             K=0.25 : 최대 로그충격 크기
#             a=6.0  : 중간 확률대의 반응 기울기
#             q0=0.5 : 수요 반응이 가장 민감한 기준 강수확률
#
#  설계 해석
#   · rain_prob_high 플래그로 게이트하지 않는다. 실제 강수확률 q 자체가 연속적인 강도를
#     담으므로 대조군의 평상시 확률 0.05 에도 작은 충격이 적용된다.
#   · 이벤트 순효과는 같은 처치군 경기에서 실제 q(ON)와 평상시 q=0.05(OFF)를 비교한다.
#     날짜·요일·월은 고정되므로 달력효과가 정확히 상쇄된다. ([A-11])
#   · 충격의 성격은 '이벤트형'이다. 예보는 경기별로 갱신되므로 시간감쇠 A(d) 를 두지
#     않고 경기별 rain_prob 자체가 강도를 담당한다. (시나리오 2와의 대비점)
#   · 취소·환불 경로. 이 모형은 M_t 감소만 다루고 우천 취소로 인한 매출 소멸은 다루지
#     않는다. rain_at_game_time 은 사후 정보라 삭제됐으므로 예보 기반 수요감소만 모형화
#     한다는 점을 보고서에 한계로 명시한다.
#   · 달력 편중. 대조군 주말 4/6 (delta_cal +0.100) vs 처치군 주말 1/6 (-0.102) 로
#     요일 구성이 기울어 있다. 순효과는 반드시 쌍체 반사실로 산출한다.
#
#  K 선정 — 2단계 스크리닝  ※ 0.40 과 0.25 는 같은 층위의 후보값이 아니다
#   · [1단계] 이론적 방어선  K <= 0.40
#     고정 SHOCK_CLIP 하한은 -0.6이고 시나리오 3 처치군의 최악 달력값은 -0.17(9월 주중)
#     이므로 강수충격이 쓸 수 있는 이론적 여유는 -0.43이다. delta_rain 이 -0.4보다
#     낮아지면 합산 log_shock 이 하한에 접근·접촉해 설계한 강수효과가 잘리고 시나리오
#     비교가 왜곡된다. 즉 0.40 은 '넘으면 안 되는 선'이지 채택 후보가 아니다.
#   · [2단계] 실증 스크리닝  K = 0.25  <- 채택값
#     clip 이 걸리기 전에도 최적가격이 0.7x/1.5x 경계에 붙는 코너해가 먼저 생길 수 있다.
#     따라서 1단계를 통과한 구간을 조직위 6경기에 직접 돌려 확인한다. [A-11] 민감도에서
#     K=0.30 은 corner hit 2건으로 정량 인용이 불가했고, K=0.25 는 clip 0건·corner hit
#     0건이다. 이론 상한이 아니라 실증적으로 안전한 0.25 를 기준안으로 채택한다. ([A-11])
# ==============================================================================
RAIN = {
    'max_shock':  0.25,   # K: 로지스틱 최대 로그충격 크기
    'steepness':  6.0,    # a: 중간 확률대 반응 기울기
    'midpoint':    0.5,    # q0: 로지스틱 중심 강수확률
    'normal_prob': 0.05,   # 식(3-2): 강수예보 OFF의 평상시 기준확률
}


def _rain_probability(row):
    """강수확률을 읽고 유한한 0~1 실수인지 검증한다."""
    q = row.get('rain_prob', None)
    if q is None or pd.isna(q):
        return None
    try:
        q = float(q)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'rain_prob은 0~1 사이 숫자여야 합니다: {q!r}') from exc
    if not np.isfinite(q) or not 0.0 <= q <= 1.0:
        raise ValueError(f'rain_prob은 0~1 사이 숫자여야 합니다: {q!r}')
    return q


def delta_rain(row, p=RAIN):
    """[시나리오 3] 강수확률 로지스틱 로그충격. 식(3-1)"""
    q = _rain_probability(row)
    if q is None:
        return 0.0
    K, a, q0 = p['max_shock'], p['steepness'], p['midpoint']
    if K < 0 or a <= 0 or not 0.0 <= q0 <= 1.0:
        raise ValueError(f'RAIN 파라미터 범위가 잘못되었습니다: {p!r}')
    return -K / (1.0 + np.exp(-a * (q - q0)))

In [47]:
# ==============================================================================
# [P-5] [전 시나리오 공통] 통합 진입점 · 쌍체 반사실 헬퍼 · 공통 상수
# ==============================================================================
#  · indicator(row, p)          : 시나리오 2 민감도용 INJURY 파라미터를 주입하는 진입점
#  · my_virtual_indicators(row) : 출제자 파이프라인이 부르는 기본 진입점 (기준 파라미터)
#  · paired_counterfactual(...) : 세 시나리오가 공유하는 동일경기 쌍체 반사실 계산기
#
#  네 지표는 모두 로그척도이므로 단순 합산하며, 고정 파이프라인 내부에서
#  M_t=M0*exp(log_shock)로 시장규모의 곱셈 수요배수로 변환된다.
#
#  [주의] run_evaluation_pipeline 의 기본값 indicator_fn 은 출제자 고정 셀 8에서
#  정의 시점의 함수 객체에 바인딩된다. 아래에서 my_virtual_indicators 를 재정의해도
#  기본값은 바뀌지 않으므로, 참가자 지표를 쓰려면 indicator_fn 을 명시적으로 넘긴다.
#  (셀 9~13의 demo 출력은 출제자 원본 라이벌 지표 기준이며, 위에서 아래로 순서대로
#   실행하는 한 그대로 재현된다.)
# ==============================================================================
def indicator(row, p=INJURY):
    """[공통] 네 로그지표를 합산한다. p는 시나리오 2의 INJURY 사양만 교체한다."""
    return (delta_calendar(row)          # [공통]       일괄적용
            + delta_star_record(row)     # [시나리오 1] 구현 완료
            + delta_injury(row, p)       # [시나리오 2] 구현 완료
            + delta_rain(row))           # [시나리오 3] 구현 완료


def my_virtual_indicators(row, cfg=CONFIG):
    """출제자 파이프라인이 호출하는 기준안 진입점.

    cfg 인자는 배포본 함수 호출 규격과의 호환성을 위해 유지하며 참가자 지표 계산에는
    사용하지 않는다. 실제 부상 기준값은 전역 INJURY 사전에서 명시적으로 주입한다.
    """
    return indicator(row, INJURY)


def paired_counterfactual(scenario_prefix, event_col, indicator_fn=None, off_value=0):
    """[공통] 동일경기 쌍체 반사실.

    처치군 경기를 그대로 두고 event_col 만 off_value/원래 값으로 바꿔 두 번 계산한다.
    시나리오 1·2는 플래그를 0으로, 시나리오 3은 rain_prob을 평상시 0.05로 둔다.
    날짜·요일·월·
    상대팀·순위가 완전히 같은 행이므로 달력 성분이 양쪽에 동일하게 들어가 정확히
    상쇄되고, 차이는 정의상 해당 이벤트 하나뿐이다.
    반환: (비교표, 처치군 원본 행)
    """
    # 별도 함수가 없으면 실제 운영 기준안(my_virtual_indicators)을 사용한다.
    fn = indicator_fn if indicator_fn is not None else my_virtual_indicators
    # 조직위 원자료의 대조군은 사용하지 않고, 해당 시나리오의 처치군 경기만 복사한다.
    g = df_games_scenarios[df_games_scenarios['scenario'].str.startswith(scenario_prefix)]
    treat = g[g['pair'] == '처치군'].reset_index(drop=True)
    treat_off = treat.copy()
    treat_off[event_col] = off_value                # 반사실: 같은 경기, 이벤트만 기준상태

    # ON/OFF 모두 같은 고정 수요모형·희소성 함수·SLSQP 엔진으로 다시 최적화한다.
    # 따라서 가격도 각 상태에 맞게 재최적화되며 수요만 사후 조정한 결과가 아니다.
    on = run_evaluation_pipeline(treat, M0_df, CAPACITY_df, indicator_fn=fn, verbose_clip=False)
    off = run_evaluation_pipeline(treat_off, M0_df, CAPACITY_df, indicator_fn=fn, verbose_clip=False)

    out = treat[['date', 'day_of_week']].copy()
    out['cal'] = treat.apply(delta_calendar, axis=1)
    for c in ['p_D7_G', 'p_D7_P', 'p_D3_G', 'p_D3_P', 'audience', 'revenue']:
        out[c + '_off'], out[c + '_on'] = off[c].to_numpy(), on[c].to_numpy()
    out['aud_%'] = (out['audience_on'] / out['audience_off'] - 1) * 100
    out['rev_%'] = (out['revenue_on'] / out['revenue_off'] - 1) * 100
    # 다음 세 열은 효과 크기가 아니라 그 효과를 정량 인용해도 되는지 판정하는 품질지표다.
    out['clipped'] = on['clipped'].to_numpy() | off['clipped'].to_numpy()       # 어느 쪽이든 clip
    out['corner'] = on['corner_hits'].to_numpy() + off['corner_hits'].to_numpy() # 가격 경계 접촉 수
    out['success'] = on['success'].to_numpy() & off['success'].to_numpy()       # 양쪽 모두 수렴
    return out, treat


# 조직위 시나리오 데이터셋은 3개 시나리오 모두 홈팀이 Jaguars 다. 분석 셀에서 공용한다.
_M0 = M0_df.query("team=='Jaguars' and season==2024")['M0'].values[0]
_CAP = CAPACITY_df.query("team=='Jaguars' and season==2024")['capacity'].values[0]
print(f'[공통] 홈팀 Jaguars  M0 = {_M0:,}   capacity = {_CAP:,}')

[공통] 홈팀 Jaguars  M0 = 17,575   capacity = 23,750


In [48]:
# ==============================================================================
# 13. [출제자 고정 영역] Phase 3 실제 제시 시나리오 데이터셋 (참가자 배포용)
#
#  시나리오별 규모
#   1. 긍정 (스타 대기록)    : 대조 4경기 + 처치 4경기 (7일 홈 4연전)
#   2. 부정I (핵심타자 부상)  : 대조 5경기 + 처치 5경기 (2개월 구간 대표 샘플)
#   3. 부정II (기습장마 예보) : 대조 6경기 + 처치 6경기 (1~2주 구간 대표 샘플)
#
#  컬럼 설명
#   - scenario: 3개 시나리오 구분
#   - pair: 대조군(비교 기준 경기) / 처치군(이벤트가 실제로 걸린 경기)
#   - date, season: 경기 날짜, 시즌
#   - home_team, away_team: 홈/원정팀
#   - day_of_week: 요일
#   - home_rank, away_rank: 홈/원정팀 순위
#   - is_star_record / is_key_injury / rain_prob_high : 해당 구간에 속하는지 여부 플래그
#     (사실관계 — 조직위 고정). 구간 내 여러 경기에 동일하게 1로 표시된다.
#   - hr_from_record    : [시나리오1] 대기록까지 남은 홈런 수 (작을수록 임박, 0=달성).
#                          ※ 이 값은 경기 결과를 사후 반영한 수치가 아니라, 각 경기가
#                          시작되기 전 이미 언론·구단 발표 등을 통해 공표되어 있던
#                          잔여 홈런 수(pregame 공표치)라는 조직위 설정(시나리오 가정)이다.
#                          즉 D-7/D-3 가격 결정 시점에 참가자가 알 수 있는 사전 정보로
#                          간주하고 사용한다.
#   - days_since_injury  : [시나리오2] 부상 발생 후 경과일 (작을수록 최근). 부상 발생일
#                          이라는 고정 사건 기준 달력상 계산값이다.
#   - rain_prob          : [시나리오3] 해당 경기의 예보 강수확률 (연속값 0~1). 가격 결정
#                          시점 기준 최신 기상예보치이다.
#
#  ※ temp_at_game_time(경기 시각 기온), rain_at_game_time(경기 당일 실제 강수 여부)은
#    모두 경기가 실제로 열려야 확정되는 당일 실측치로, D-7/D-3 가격 결정 시점에는
#    관측이 불가능한 사후 정보다. 시나리오 관련성과 무관하게 전 시나리오 공통으로
#    참가자 배포용 데이터셋에서 컬럼 자체를 삭제한다.
#
#  참가자는 flag(0/1)만 보고 구간 전체에 동일한 δ를 적용(일괄적용)할 수도 있고,
#  hr_from_record / days_since_injury / rain_prob 같은 연속값으로 경기별 강도를 달리
#  반영(선별적용)할 수도 있다. 어느 쪽을 택하든 그 근거를 서술하는 것이 채점 대상이며,
#  baseline은 이 3개 시나리오를 풀어주지 않는다.
# ==============================================================================
df_games_scenarios = pd.DataFrame([
    # ==== 시나리오 1: 긍정 (스타 선수 홈런 대기록 임박) — 대조 4 + 처치 4 ====
    # 대조군: 시즌 초반, 대기록 이슈 없는 평범한 4연전
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-05', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-06', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-07', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '대조군', 'date': '2024-04-08', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    # 처치군: 7일 뒤 홈 4연전, 대기록까지 3->0개로 카운트다운 (마지막 경기에서 달성)
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-14', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 3, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-15', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 2, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-16', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 1, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '1. 긍정 (스타 대기록)', 'pair': '처치군', 'date': '2024-06-17', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 1, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': 0, 'days_since_injury': None, 'rain_prob': None},

    # ==== 시나리오 2: 부정I (핵심 타자 부상 이탈, 2개월) — 대조 5 + 처치 5 ====
    # 대조군: 부상 이슈 없는 시즌 초반 대표 샘플 5경기
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-23', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-24', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-30', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-03-31', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '대조군', 'date': '2024-04-13', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': None},
    # 처치군: 부상(2024-07-01) 이후 2개월 구간, 경과일 2/14/30/45/58일 대표 샘플
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-07-03', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 2, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-07-15', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 14, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-07-31', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 30, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-08-15', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '목',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 45, 'rain_prob': None},
    {'scenario': '2. 부정I (핵심타자 부상)', 'pair': '처치군', 'date': '2024-08-28', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 1, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': 58, 'rain_prob': None},

    # ==== 시나리오 3: 부정II (기습 장마 예보, 1~2주) — 대조 6 + 처치 6 ====
    # 대조군: 맑은 날씨가 이어지는 시즌 중반 대표 샘플 6경기
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-19', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-20', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-21', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-26', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '금',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-27', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '토',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '대조군', 'date': '2024-04-28', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 0,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.05},
    # 처치군: 9/1 기습 예보 이후 1~2주 구간, 강수확률이 0.85 -> 0.20으로 점차 낮아짐
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-02', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.85},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-03', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '화',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.75},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-04', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '수',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.65},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-08', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '일',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.50},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-09', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '월',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.35},
    {'scenario': '3. 부정II (기습장마 예보)', 'pair': '처치군', 'date': '2024-09-10', 'season': 2024,
     'home_team': 'Jaguars', 'away_team': 'Falcons', 'day_of_week': '화',
     'home_rank': 2, 'away_rank': 5,
     'is_star_record': 0, 'is_key_injury': 0, 'rain_prob_high': 1,
     'hr_from_record': None, 'days_since_injury': None, 'rain_prob': 0.20}
])
df_games_scenarios

,scenario,pair,date,season,home_team,away_team,day_of_week,home_rank,away_rank,is_star_record,is_key_injury,rain_prob_high,hr_from_record,days_since_injury,rain_prob
0,1. 긍정 (스타 대기록),대조군,2024-04-05,2024,Jaguars,Falcons,금,2,5,0,0,0,NaN,NaN,NaN
1,1. 긍정 (스타 대기록),대조군,2024-04-06,2024,Jaguars,Falcons,토,2,5,0,0,0,NaN,NaN,NaN
2,1. 긍정 (스타 대기록),대조군,2024-04-07,2024,Jaguars,Falcons,일,2,5,0,0,0,NaN,NaN,NaN
3,1. 긍정 (스타 대기록),대조군,2024-04-08,2024,Jaguars,Falcons,월,2,5,0,0,0,NaN,NaN,NaN
4,1. 긍정 (스타 대기록),처치군,2024-06-14,2024,Jaguars,Falcons,금,2,5,1,0,0,3.0,NaN,NaN
5,1. 긍정 (스타 대기록),처치군,2024-06-15,2024,Jaguars,Falcons,토,2,5,1,0,0,2.0,NaN,NaN
6,1. 긍정 (스타 대기록),처치군,2024-06-16,2024,Jaguars,Falcons,일,2,5,1,0,0,1.0,NaN,NaN
7,1. 긍정 (스타 대기록),처치군,2024-06-17,2024,Jaguars,Falcons,월,2,5,1,0,0,0.0,NaN,NaN
8,2. 부정I (핵심타자 부상),대조군,2024-03-23,2024,Jaguars,Falcons,토,2,5,0,0,0,NaN,NaN,NaN
9,2. 부정I (핵심타자 부상),대조군,2024-03-24,2024,Jaguars,Falcons,일,2,5,0,0,0,NaN,NaN,NaN


---
## [실행] df_games_scenarios 파이프라인 실행

`df_games_demo`용 cell 10~12(게임명 문자열로 'Rams'/'Falcons'를 찾는 방식)는 `df_games_scenarios`
구조(여러 시나리오 × 여러 경기)에 맞지 않는다. 대신 `scenario`/`pair` 컬럼으로 그룹핑해서 보여주는
별도 실행 함수를 아래에 둔다.

본인이 작성한 `my_virtual_indicators()`를 다 채운 뒤, 아래 셀을 그대로 실행하면 시나리오별
경기 단위 결과와 대조군-처치군 비교 요약이 출력된다. 지금은 라이벌전 지표만 구현돼 있어
3개 시나리오 전부 동일한 결과(log_shock=0)가 나오는 게 정상이다 — 본인의 지표를 추가하면서
이 셀을 다시 실행해 값이 어떻게 바뀌는지 직접 확인하는 용도로 쓰면 된다.

In [49]:
# ==============================================================================
# 14. [실행] Phase 3 시나리오 데이터셋(df_games_scenarios) 파이프라인 실행
# ==============================================================================
def run_scenario_pipeline(indicator_fn=my_virtual_indicators):
    """df_games_scenarios를 run_evaluation_pipeline에 태우고,
    scenario/pair 컬럼을 붙여 시나리오별로 읽기 좋게 정리해서 돌려준다."""
    res = run_evaluation_pipeline(df_games_scenarios, M0_df, CAPACITY_df, indicator_fn=indicator_fn)
    res = res.reset_index(drop=True)
    meta = df_games_scenarios[['scenario', 'pair', 'date']].reset_index(drop=True)
    out = pd.concat([meta, res.drop(columns=['capacity'])], axis=1)  # M0는 유지, capacity만 제거
    return out


def print_scenario_results(indicator_fn=my_virtual_indicators, label='my_virtual_indicators'):
    """cell 10([실행] 시나리오 시뮬레이션)과 동일한 포맷으로 df_games_scenarios 결과를 출력한다."""
    res = run_scenario_pipeline(indicator_fn)

    bG, bP = BASE_PRICES['일반석'], BASE_PRICES['프리미엄석']
    view = res.copy()
    for c, b in [('p_D7_G', bG), ('p_D7_P', bP), ('p_D3_G', bG), ('p_D3_P', bP)]:
        view[c + '_x'] = (view[c] / b).round(2)

    print('=' * 100)
    print(f' [{label}] df_games_scenarios 시뮬레이션  —  M_t = M_0 * exp(log_shock)')
    print('=' * 100)
    print(view[['scenario', 'pair', 'date', 'M0', 'raw_log_shock', 'log_shock', 'shock_x', 'M_t',
                'p_D7_G_x', 'p_D7_P_x', 'p_D3_G_x', 'p_D3_P_x',
                'audience', 'revenue', 'corner_hits']].to_string(index=False))
    n_clipped = int(view['clipped'].sum())
    if n_clipped > 0:
        print(f"\n※ 위 표에서 raw_log_shock(설계한 원래 값)과 log_shock(SHOCK_CLIP 적용 후 값)이"
              f" 다른 행이 {n_clipped}개 있습니다 — 해당 행은 clip되어 delta 총합이 잘렸습니다.")

    print()
    print('-' * 100)
    print(' [시나리오별 대조군 vs 처치군 비교 (구간 평균 기준)]')
    print('-' * 100)
    summary = res.groupby(['scenario', 'pair'])['revenue'].mean().unstack()
    for scenario in summary.index:
        rev_control = summary.loc[scenario, '대조군']
        rev_treat = summary.loc[scenario, '처치군']
        idx = rev_treat / rev_control * 100
        pct = (rev_treat / rev_control - 1) * 100
        print(f"  {scenario:<22} 대조군 {rev_control:>15,.0f}원 (지수 100.0)  ->  "
              f"처치군 {rev_treat:>15,.0f}원 (지수 {idx:6.1f}, {pct:+.1f}%)")
    print('-' * 100)

    return res


res_scenarios = print_scenario_results(my_virtual_indicators, 'my_virtual_indicators')

 [my_virtual_indicators] df_games_scenarios 시뮬레이션  —  M_t = M_0 * exp(log_shock)
         scenario pair       date    M0  raw_log_shock  log_shock  shock_x   M_t  p_D7_G_x  p_D7_P_x  p_D3_G_x  p_D3_P_x  audience   revenue  corner_hits
   1. 긍정 (스타 대기록)  대조군 2024-04-05 17575         0.1000     0.1000    1.105 19423      1.30      1.14      1.40      1.21     12061 285788758            0
   1. 긍정 (스타 대기록)  대조군 2024-04-06 17575         0.1000     0.1000    1.105 19423      1.30      1.14      1.40      1.21     12061 285788758            0
   1. 긍정 (스타 대기록)  대조군 2024-04-07 17575         0.1000     0.1000    1.105 19423      1.30      1.14      1.40      1.21     12061 285788758            0
   1. 긍정 (스타 대기록)  대조군 2024-04-08 17575        -0.3100    -0.3100    0.733 12890      1.42      1.03      1.47      1.09      8033 210807262            0
   1. 긍정 (스타 대기록)  처치군 2024-06-14 17575         0.3311     0.3311    1.392 24472      1.26      1.19      1.39      1.27     15224 341239406         

# [참가자 작성 영역 — 추가 분석 및 검증]

아래 셀은 참가자가 정의한 가상지표의 근거 재현, 쌍체 반사실, 민감도 및 수치 검증만 수행합니다. 바로 위 출제자 고정 셀의 함수·데이터·실행 코드는 수정하지 않습니다.


In [50]:
# ==============================================================================
# [A-1] [공통] 달력 지표 검산 — KBO 원자료에서 계수까지 재현
# ==============================================================================
#  상세 분석 순서: 공통([A-1]~[A-2]) -> 시나리오 1([A-3])
#                 -> 시나리오 2([A-4]~[A-10]) -> 시나리오 3([A-11])
#  조직위 대조군/처치군 원자료 비교는 달력+이벤트의 정책 총효과로 해석하고,
#  이벤트 순효과는 동일경기 쌍체 반사실로 분리한다.
#
#  근거: KBO 공식 기록 2023·2024 시즌 (9/15 기준)
#        표3 월별 경기수·총관중 / 표4 요일별 평균관중
#  아래 KBO_MONTH_RAW 는 표3 원자료 그대로다. 비율·delta 를 하드코딩하지 않고
#  원자료에서 산출해 보고서 수치와 코드가 일대일로 대응되게 한다.  -> [P-1] 식(0-1)(0-2)
# ==============================================================================
KBO_MONTH_RAW = {                    # {연도: {월: (경기수, 총관중)}}
    2024: {3: (37, 574_560), 4: (121, 1_604_664), 5: (122, 1_815_455),
           6: (125, 1_904_507), 7: (97, 1_438_738), 8: (119, 1_834_055),
           9: (50, 848_779)},
    2023: {4: (120, 1_265_432), 5: (113, 1_299_684), 6: (121, 1_353_843),
           7: (83, 938_815), 8: (113, 1_162_350), 9: (110, 1_261_466)},
}
KBO_DOW_MEAN = {                     # {연도: (주말 금토일 평균, 주중 화수목 평균, 전체 평균)}
    2024: (17_321, 12_627, 14_934),
    2023: (13_788, 8_345, 11_233),   # 요일표의 전체 평균은 월별표(11,033)와 표본이 다르다
}

# --- 1단계: 각 해 리그 평균 대비 상대비율 --------------------------------------
#   연도 절대상승(리그 인기 추세)을 제거한다. 그 추세는 M0 가 이미 반영한다.
rows = []
for yr, d in KBO_MONTH_RAW.items():
    season_mean = sum(t for _, t in d.values()) / sum(g for g, _ in d.values())
    for m, (g, t) in d.items():
        rows.append({'year': yr, 'month': m, 'games': g, 'mean': t / g,
                     'ratio': (t / g) / season_mean})
kbo = pd.DataFrame(rows)

print('=' * 96)
print(' [A-1] 1단계 — KBO 원자료 -> 월별 상대비율')
print('=' * 96)
piv = kbo.pivot(index='month', columns='year', values='ratio')
piv['2개년 평균'] = piv.mean(axis=1)
print(piv.round(4).to_string())
for yr, d in KBO_MONTH_RAW.items():
    G, T = sum(g for g, _ in d.values()), sum(t for _, t in d.values())
    print(f'  {yr} 계 {G}경기 {T:,}명 -> 시즌평균 {T/G:,.0f}명')

# --- 2단계: 3구간 집약 + log 변환 ----------------------------------------------
print('\n' + '-' * 96)
print(' [A-1] 2단계 — 3구간 집약 + log 변환')
print('-' * 96)
BINS = {'성수기 3,9': [3, 9], '평수기 5~8': [5, 6, 7, 8], '비수기 4': [4]}
for lab, ms in BINS.items():
    r = np.mean([piv.loc[m, '2개년 평균'] for m in ms])
    n = int(kbo[kbo['month'].isin(ms)]['games'].sum())
    print(f'  {lab:<11} 구간 평균비율 {r:.4f}   ln = {np.log(r):+.5f}'
          f'   -> 채택 {CAL_MONTH.get(ms[0], 0.0):+.2f}   (표본 {n}경기)')
print('  ※ 평수기는 ln = +0.007 로 사실상 0이다. 반올림 결과가 아니라 "평수기를 시즌')
print('    평균 기준선으로 둔다"는 설계 선택이며, 그래서 0.00 을 부여했다.')

print()
for yr, (we, wd, tot) in KBO_DOW_MEAN.items():
    print(f'  {yr}  주말 {we:,}/{tot:,} = {we/tot:.4f}   주중 {wd:,}/{tot:,} = {wd/tot:.4f}')
_we = np.mean([we / tot for we, _, tot in KBO_DOW_MEAN.values()])
_wd = np.mean([wd / tot for _, wd, tot in KBO_DOW_MEAN.values()])
print(f'  주말(금토일) 2개년 평균비율 {_we:.4f}  ln = {np.log(_we):+.5f}  -> 채택 {CAL_WEEKEND:+.2f}')
print(f'  주중(화수목) 2개년 평균비율 {_wd:.4f}  ln = {np.log(_wd):+.5f}  -> 채택 {CAL_WEEKDAY:+.2f}')
print('  ※ KBO 는 월요일 경기가 없어 월요일 편성분은 주중(화수목) 계수로 대체한다.')

# --- 중심화 점검 ---------------------------------------------------------------
#   M0 는 이미 평균적인 요일·월 구성을 반영한다. 지표 평균이 0에서 멀면 이중계상이다.
print('\n' + '-' * 96)
print(' [A-1] 중심화 점검')
print('-' * 96)
for yr in KBO_MONTH_RAW:
    s = kbo[kbo['year'] == yr]
    lvl = (s['ratio'] * s['games']).sum() / s['games'].sum()
    lg = (np.log(s['ratio']) * s['games']).sum() / s['games'].sum()
    print(f'  월 지표 {yr}   level 경기가중 평균 {lvl:.6f}   log 경기가중 평균 {lg:+.6f}')
print('    -> level 이 정확히 1.000000 이다. 시즌평균이 월평균의 경기가중 평균이므로')
print('       구성상 항등이며, 월 지표는 완전히 중심화돼 있다. log 오목성 손실은 -0.002.')

# 요일은 경기수가 표에 없으므로 평균관중 항등식에서 주말 편성비중을 역산한다
for yr, (we, wd, tot) in KBO_DOW_MEAN.items():
    w = (tot - wd) / (we - wd)
    print(f'  요일 지표 {yr}   주말 편성비중 역산 {w:.4f}   (화~일 6일 편성의 3/6 = 0.50 과 정합)')
_w = np.mean([(tot - wd) / (we - wd) for we, wd, tot in KBO_DOW_MEAN.values()])
_off = _w * np.log(_we) + (1 - _w) * np.log(_wd)
print(f'    -> log 편성가중 평균 {_off:+.4f}. 요일 지표에만 약 -0.02 하방 오프셋이 남는다.')
print('       전 경기에 동일하게 걸리므로 군간 비교에서는 상쇄되고 절대 M_t 만 소폭 낮아진다.')

print('\n  [표본 주의] 2023 시즌은 4월 개막이라 3월 자료가 없다. 성수기 구간의 3월은')
print('    2024 단독 37경기, 9월은 2개년 160경기다. 성수기 +0.06 의 근거 두께가 다르므로')
print('    보고서에는 "9월 주도, 3월은 단일연도"임을 명시한다.')

 [A-1] 1단계 — KBO 원자료 -> 월별 상대비율
year     2023    2024  2개년 평균
month                        
3         NaN  1.0398  1.0398
4      0.9558  0.8880  0.9219
5      1.0425  0.9964  1.0195
6      1.0141  1.0202  1.0172
7      1.0252  0.9932  1.0092
8      0.9323  1.0320  0.9822
9      1.0394  1.1367  1.0881
  2024 계 671경기 10,020,758명 -> 시즌평균 14,934명
  2023 계 660경기 7,281,590명 -> 시즌평균 11,033명

------------------------------------------------------------------------------------------------
 [A-1] 2단계 — 3구간 집약 + log 변환
------------------------------------------------------------------------------------------------
  성수기 3,9     구간 평균비율 1.0639   ln = +0.06198   -> 채택 +0.06   (표본 197경기)
  평수기 5~8     구간 평균비율 1.0070   ln = +0.00699   -> 채택 +0.00   (표본 893경기)
  비수기 4       구간 평균비율 0.9219   ln = -0.08130   -> 채택 -0.08   (표본 241경기)
  ※ 평수기는 ln = +0.007 로 사실상 0이다. 반올림 결과가 아니라 "평수기를 시즌
    평균 기준선으로 둔다"는 설계 선택이며, 그래서 0.00 을 부여했다.

  2024  주말 17,321/14,934 = 1.1598   주중 12,627/14,934 = 0.8455
  2023  주말 13

In [51]:
# ==============================================================================
# [A-2] [공통] 시나리오별 달력 편중 · SHOCK_CLIP 예산
# ==============================================================================
#  달력 지표는 일괄적용이라 대조군에도 걸린다. 세 시나리오 각각에서 처치군-대조군
#  달력 격차가 얼마인지 먼저 확인해야 원자료 비교의 해석 가능 여부를 판정할 수 있다.
# ==============================================================================
g_all = df_games_scenarios.copy()
g_all['cal'] = g_all.apply(delta_calendar, axis=1)
tab = g_all.groupby(['scenario', 'pair'])['cal'].mean().unstack()
tab['격차'] = tab['처치군'] - tab['대조군']
tab['수요배수'] = np.exp(tab['격차'])
print('=' * 96)
print(' [A-2] 시나리오별 달력 편중 (처치군 - 대조군)')
print('=' * 96)
print(tab.round(4).to_string())
print('  → 시나리오 2는 달력만으로 처치군 수요배수가 약 36% 낮다. 부상 기준안의')
print('    순수 수요충격(약 4%)보다 훨씬 크므로 원자료 군간 비교를 부상효과로 읽을 수 없다.')
print('    따라서 이벤트 순효과는 대조군 비교가 아니라 쌍체 반사실로 산출한다. ([A-4])')
print('    시나리오 1(+8%)·3(-18%)도 같은 이유로 쌍체 반사실이 필요하다.')

# --- SHOCK_CLIP 예산 ----------------------------------------------------------
#   달력 delta 는 이미 확정된 값이므로, 남은 시나리오 지표가 쓸 수 있는 여유가 정해진다.
#   clip 이 발동하면 설계한 delta 가 잘려 시나리오 간 비교가 왜곡되므로 사전에 확인한다.
#   [주의] 이론적 최소(4월 주중 -0.31)는 조직위 데이터에 존재하지 않는 조합이다.
#          시나리오 3 대조군은 4월이지만 전부 금토일이라 cal = +0.10 이다.
#          따라서 실제 경기별 여유로 예산을 잡는다.
print('\n' + '-' * 96)
print(f' [A-2] SHOCK_CLIP 예산  허용 {SHOCK_CLIP}')
print('-' * 96)
g_all['여유_하한'] = SHOCK_CLIP[0] - g_all['cal']    # 시나리오 지표가 이보다 작아지면 clip
g_all['여유_상한'] = SHOCK_CLIP[1] - g_all['cal']
bud = g_all.groupby('scenario').agg(
    cal_min=('cal', 'min'), cal_max=('cal', 'max'),
    음의여유=('여유_하한', 'max'), 양의여유=('여유_상한', 'min'))
print(bud.round(4).to_string())
print('  음의여유 = 그 시나리오에서 가장 빡빡한 경기의 시나리오 지표 하한')
print('  양의여유 = 가장 빡빡한 경기의 시나리오 지표 상한')
print('\n  ※ 조정이 필요하면 달력 delta 가 아니라 시나리오 delta 를 조정한다. 달력은 KBO')
print('    원자료 근거가 있고 시나리오 지표는 참가자 설계 영역이므로, 근거 있는 값을')
print('    clip 때문에 흔들면 정성평가에서 약점이 된다.')

 [A-2] 시나리오별 달력 편중 (처치군 - 대조군)
pair                  대조군     처치군      격차    수요배수
scenario                                         
1. 긍정 (스타 대기록)    -0.0025  0.0775  0.0800  1.0833
2. 부정I (핵심타자 부상)   0.2120 -0.2300 -0.4420  0.6427
3. 부정II (기습장마 예보)  0.1000 -0.1017 -0.2017  0.8174
  → 시나리오 2는 달력만으로 처치군 수요배수가 약 36% 낮다. 부상 기준안의
    순수 수요충격(약 4%)보다 훨씬 크므로 원자료 군간 비교를 부상효과로 읽을 수 없다.
    따라서 이벤트 순효과는 대조군 비교가 아니라 쌍체 반사실로 산출한다. ([A-4])
    시나리오 1(+8%)·3(-18%)도 같은 이유로 쌍체 반사실이 필요하다.

------------------------------------------------------------------------------------------------
 [A-2] SHOCK_CLIP 예산  허용 (-0.6, 0.8)
------------------------------------------------------------------------------------------------
                   cal_min  cal_max  음의여유  양의여유
scenario                                       
1. 긍정 (스타 대기록)       -0.31     0.18 -0.29  0.62
2. 부정I (핵심타자 부상)     -0.23     0.24 -0.37  0.56
3. 부정II (기습장마 예보)    -0.17     0.24 -0.43  0.56
  음의여유 = 그 시나리오에서 가장 빡빡한 경기의 시나리오 지표 하한
  양의여유 = 가장

In [52]:
# ==============================================================================
# [A-3] [시나리오 1] 스타 대기록 — 달성확률 · clip · 쌍체 반사실 점검
# ==============================================================================
#  조직위 처치군 4경기의 hr_from_record=3->2->1->0을 그대로 사용한다.
#  0은 해당 경기에서 기록을 달성하는 경우이므로 P_moment=1이다.
#  아래의 기준안 결과(res1)는 실제 가격 제안과 보고서 본문에 사용하는 값이다.
#  민감도 분석(sens1)은 기준 파라미터 STAR를 바꾸지 않는 별도의 검증 절차이며,
#  체이스 화제성·목격 프리미엄·파크팩터를 한 축씩만 바꾸는 OAT 방식으로 수행한다.
#  여러 파라미터를 동시에 바꾸지 않는 이유는 결과 변화가 어느 가정에서 발생했는지
#  분리해서 해석하고, 기준안의 결론(관중·수익 증가)이 합리적 범위에서 유지되는지
#  확인하기 위해서다.
# ==============================================================================
def scenario1_indicator(row, p=STAR):
    """시나리오 1 민감도용 지표 진입점. 다른 지표는 기준안으로 고정한다."""
    return (delta_calendar(row) + delta_star_record(row, p)
            + delta_injury(row) + delta_rain(row))


def run_scenario1(p=STAR):
    fn = lambda row: scenario1_indicator(row, p)
    return paired_counterfactual('1.', 'is_star_record', fn)


s1 = df_games_scenarios[df_games_scenarios['scenario'].str.startswith('1.')].copy()
s1['cal'] = s1.apply(delta_calendar, axis=1)
s1['p_moment'] = s1.apply(
    lambda r: _p_moment(r['hr_from_record']) if int(r['is_star_record']) == 1 else 0.0,
    axis=1)
s1['delta_star'] = s1.apply(delta_star_record, axis=1)
s1['total_delta'] = s1['cal'] + s1['delta_star']
print('=' * 96)
print(' [A-3] 시나리오 1 데이터 구성 · 달성확률 · 로그충격')
print('=' * 96)
print(s1[['pair', 'date', 'day_of_week', 'hr_from_record', 'p_moment',
          'cal', 'delta_star', 'total_delta']].round(4).to_string(index=False))
print(f"\n  달력 격차 {s1[s1['pair']=='처치군']['cal'].mean() - s1[s1['pair']=='대조군']['cal'].mean():+.4f}"
      f"  -> 처치군이 달력만으로 유리하다. 원자료 비교 불가.")

# --- 설계한 대기록 지표의 실제 SHOCK_CLIP 점검 -------------------------------
s1t = s1[s1['pair'] == '처치군']
n_clip = int(((s1t['total_delta'] < SHOCK_CLIP[0]) |
              (s1t['total_delta'] > SHOCK_CLIP[1])).sum())
print(f'\n  [clip 점검] 총 log_shock {s1t["total_delta"].min():+.4f}'
      f' ~ {s1t["total_delta"].max():+.4f}  /  SHOCK_CLIP {SHOCK_CLIP}  /  clip {n_clip}건')

# --- 동일경기 쌍체 반사실 — 날짜·요일·월을 고정하고 대기록 플래그만 OFF/ON -------
res1, _ = run_scenario1()
res1['hr_from_record'] = s1t['hr_from_record'].to_numpy()
res1['p_moment'] = s1t['p_moment'].to_numpy()
res1['delta_star'] = s1t['delta_star'].to_numpy()
print('\n' + '-' * 96)
print(' [A-3] 쌍체 반사실 — 대기록 OFF / ON')
print('-' * 96)
print(res1[['date', 'day_of_week', 'hr_from_record', 'p_moment', 'cal', 'delta_star',
            'audience_off', 'audience_on', 'aud_%',
            'revenue_off', 'revenue_on', 'rev_%']].round(4).to_string(index=False))
print(f"\n  4경기 평균  기대관중 {res1['aud_%'].mean():+.2f}%   기대수익 {res1['rev_%'].mean():+.2f}%")
print('  ※ 동일 경기의 달력 성분은 OFF/ON 양쪽에 같게 들어가므로 위 비율은 대기록 순효과다.')

# --- 검증용 민감도 분석: 기준안과 분리하여 한 축씩 변경(OAT) --------------------
#  각 tuple의 두 번째 원소는 STAR 기준안에서 덮어쓸 값만 담는다.
#  빈 dict인 '기준안'은 민감도 표 안에서 운영 기준값과 양 끝값을 직접 비교하기 위한 행이다.
#  d_chase   : 기록 추격 자체의 공통 화제성에 대한 보수/공격 범위
#  d_moment  : 해당 경기 달성 목격 프리미엄의 상한에 대한 보수/공격 범위
#  park_factor: 구장 환경에 따른 경기당 홈런 발생률의 낮음/높음 범위
SCENARIO1_SENSITIVITY_CASES = [
    ('체이스 0.10',      {'d_chase': 0.10}),
    ('기준안',           {}),
    ('체이스 0.20',      {'d_chase': 0.20}),
    ('목격상한 0.20',    {'d_moment': 0.20}),
    ('목격상한 0.40',    {'d_moment': 0.40}),
    ('파크팩터 0.74',    {'park_factor': 0.74}),
    ('파크팩터 1.47',    {'park_factor': 1.47}),
]

#  STAR 자체는 수정하지 않는다. 매 반복마다 새 dict를 만들어 기준안 재실행 시에도
#  항상 STAR={d_chase:0.15, d_moment:0.30, park_factor:1.02}가 유지되게 한다.
scenario1_sensitivity_rows = []
for case_label, overrides in SCENARIO1_SENSITIVITY_CASES:
    case_params = {**STAR, **overrides}
    case_result, _ = run_scenario1(case_params)
    case_deltas = s1t.apply(
        lambda row: delta_star_record(row, case_params), axis=1)
    scenario1_sensitivity_rows.append({
        'case': case_label,
        'D_CHASE': case_params['d_chase'],
        'D_MOMENT': case_params['d_moment'],
        'PARK_FACTOR': case_params['park_factor'],
        'delta_min': case_deltas.min(), 'delta_max': case_deltas.max(),
        'aud_%': case_result['aud_%'].mean(),
        'rev_%': case_result['rev_%'].mean(),
        'clipped': int(case_result['clipped'].sum()),
        'corner': int(case_result['corner'].sum()),
        'converged': bool(case_result['success'].all()),
    })

sens1 = pd.DataFrame(scenario1_sensitivity_rows)
#  usable=True인 행만 정량 범위로 인용할 수 있다. 방향이 예상과 같더라도 clip 또는
#  가격 경계해가 발생하면 고정 상·하한이 결과를 제한한 것이므로 정책 효과로 해석하지 않는다.
sens1['usable'] = (
    (sens1['clipped'] == 0) & (sens1['corner'] == 0) & sens1['converged'])
print('\n' + '-' * 104)
print(f' [A-3] 시나리오 1 민감도 ({len(SCENARIO1_SENSITIVITY_CASES)}케이스, 쌍체 반사실 기준)')
print('-' * 104)
print(sens1.round(4).to_string(index=False))
sens1_usable = sens1[sens1['usable']]
print(f"\n  유효 케이스 기대수익 변화 범위 "
      f"{sens1_usable['rev_%'].min():+.2f}% ~ {sens1_usable['rev_%'].max():+.2f}%")

#  아래 assert는 민감도 코드를 남기는 핵심 이유다. 누군가 후보값이나 고정함수를 바꿔
#  결론의 부호·clip·경계해·수렴 조건이 깨지면 노트북 실행 단계에서 즉시 드러난다.
assert (sens1['aud_%'] > 0).all() and (sens1['rev_%'] > 0).all()
assert sens1['usable'].all()
print('  [OK] 전 케이스 관중·수익 증가 / clip 0건 / corner hit 0건 / SLSQP 전건 수렴')

 [A-3] 시나리오 1 데이터 구성 · 달성확률 · 로그충격
pair       date day_of_week  hr_from_record  p_moment   cal  delta_star  total_delta
 대조군 2024-04-05           금             NaN    0.0000  0.10      0.0000       0.1000
 대조군 2024-04-06           토             NaN    0.0000  0.10      0.0000       0.1000
 대조군 2024-04-07           일             NaN    0.0000  0.10      0.0000       0.1000
 대조군 2024-04-08           월             NaN    0.0000 -0.31      0.0000      -0.3100
 처치군 2024-06-14           금             3.0    0.0035  0.18      0.1511       0.3311
 처치군 2024-06-15           토             2.0    0.0364  0.18      0.1609       0.3409
 처치군 2024-06-16           일             1.0    0.2573  0.18      0.2272       0.4072
 처치군 2024-06-17           월             0.0    1.0000 -0.23      0.4500       0.2200

  달력 격차 +0.0800  -> 처치군이 달력만으로 유리하다. 원자료 비교 불가.

  [clip 점검] 총 log_shock +0.2200 ~ +0.4072  /  SHOCK_CLIP (-0.6, 0.8)  /  clip 0건

--------------------------------------------------------------------

In [53]:
# ==============================================================================
# [A-4] [시나리오 2] 동일경기 쌍체 반사실 비교 — 부상 순효과
# ==============================================================================
#  달력 지표가 일괄적용이라 대조군 delta != 0 이다. 따라서 대조군을 반사실로 쓸 수 없다.
#  대신 처치군 5경기를 그대로 두고 is_key_injury 플래그만 0/1 로 바꿔 두 번 계산한다.
#  날짜·요일·월·상대팀·순위가 완전히 같은 행이므로 달력 성분이 양쪽에 동일하게 들어가
#  정확히 상쇄되고, 차이는 정의상 부상 하나뿐이다.  (공통 헬퍼: [P-5])
#  d=2의 손실이 d=58보다 큰 이유는 선수가 회복됐다고 가정해서가 아니라, 식(2-3)에서
#  초기 관심 부분 (1-rho)만 8일 반감기로 줄고 결장 지속효과 rho는 남기 때문이다.
#  관중보다 수익 손실이 작은 이유는 ON/OFF 각각 가격을 다시 최적화해 일부를 회수하기 때문이다.
# ==============================================================================
def run_scenario2(p=INJURY):
    """부상 OFF/ON을 각각 재최적화하고 식(2-1)~(2-6)의 중간값을 결합한다.

    반환되는 한 행은 동일한 날짜·요일·상대팀을 가진 한 경기의 OFF/ON 쌍이다.
    p에 INJURY 복사본을 넣으면 기준 함수는 유지한 채 민감도 케이스만 실행할 수 있다.
    """
    out, treat = paired_counterfactual('2.', 'is_key_injury', lambda r: indicator(r, p))
    det = treat.apply(lambda r: pd.Series(delta_injury(r, p, detail=True)), axis=1)
    out = pd.concat([out, det], axis=1)
    out.insert(2, 'days_since_injury', treat['days_since_injury'].to_numpy())
    # shock_x는 전체 달력효과를 제외한 부상만의 시장규모 배수다.
    # 예: delta=-0.05이면 shock_x=exp(-0.05)=0.951로 부상 경로 수요가 약 4.9% 감소한다.
    out['shock_x'] = np.exp(out['delta'])
    return out


res2 = run_scenario2()
print('=' * 108)
print(' [A-4] 동일경기 쌍체 반사실 — 부상 OFF / ON')
print('=' * 108)
print(res2[['date', 'day_of_week', 'days_since_injury', 'cal', 'A', 'delta', 'shock_x',
            'audience_off', 'audience_on', 'aud_%',
            'revenue_off', 'revenue_on', 'rev_%']].round(4).to_string(index=False))
print(f"\n  5경기 평균  기대관중 {res2['aud_%'].mean():+.2f}%"
      f"   기대수익 {res2['rev_%'].mean():+.2f}%")
print('  ※ cal 열이 OFF/ON 양쪽에 동일하게 들어가 상쇄되므로 이 값이 부상 순효과다.')

# OFF와 ON을 각각 SLSQP로 재최적화했으므로 아래는 같은 가격에 수요만 바꾼 결과가 아니다.
print('\n [A-4] 최적가격 — 부상 없음 -> 부상 (5경기 평균)')
for col, lab in [('p_D7_G', 'D-7 일반석'), ('p_D7_P', 'D-7 프리미엄석'),
                 ('p_D3_G', 'D-3 일반석'), ('p_D3_P', 'D-3 프리미엄석')]:
    a, b = res2[col + '_off'].mean(), res2[col + '_on'].mean()
    print(f"   {lab:<14} {a:>9,.0f}원 -> {b:>9,.0f}원   ({(b/a-1)*100:+.2f}%)")
print('   ※ 전 좌석 일괄 할인이 아니다. 일반석은 올리고 프리미엄석은 내리는 차등 재배분.')

# 가격 부호가 직관과 반대로 보이는 이유를 좌석등급별 판매량과 용량으로 직접 확인한다.
# 전체 관중/총좌석만 보면 포화를 놓칠 수 있으므로 G·P를 따로 본다. 대표 첫 경기의
# OFF/ON은 달력조건이 같고 부상 delta만 다르며, 다른 네 경기도 같은 구조를 보인다.
_m = res2.iloc[0]
_M_off = _M0 * np.exp(_m['cal'])
_M_on = _M_off * np.exp(_m['delta'])
_x_off = np.array([_m[c + '_off'] for c in ['p_D7_G', 'p_D7_P', 'p_D3_G', 'p_D3_P']])
_x_on = np.array([_m[c + '_on'] for c in ['p_D7_G', 'p_D7_P', 'p_D3_G', 'p_D3_P']])
_d_off = revenue_detail(_x_off, _M_off, _CAP)
_d_on = revenue_detail(_x_on, _M_on, _CAP)

print('\n [A-4] 가격 방향 진단 — 대표 경기 좌석등급별 판매·소진율')
for _seg, _cap_key in [('G', 'cap_G'), ('P', 'cap_P')]:
    _q0 = _d_off[f'Q_D7_{_seg}'] + _d_off[f'Q_D3_{_seg}']
    _q1 = _d_on[f'Q_D7_{_seg}'] + _d_on[f'Q_D3_{_seg}']
    _cap_seg = _d_off[_cap_key]
    print(f"   {_seg}석 판매 { _q0:>8,.1f} -> {_q1:>8,.1f}석"
          f"  / 소진율 {_q0/_cap_seg*100:>6.2f}% -> {_q1/_cap_seg*100:>6.2f}%")
print('   → 프리미엄석은 OFF부터 100% 소진되는 희소 좌석이고 일반석은 여유가 크다.')
print('     부상으로 시장이 작아지면 고정 MNL 모형은 프리미엄 물량을 지키기 위해 P 가격을')
print('     낮추고, G 가격을 올려 P의 상대매력을 높이는 교차대체를 선택한다. 일반석 판매가')
print('     감소분을 흡수한다. 가격 경계해는 아니지만 P 좌석용량 제약이 결합된 결과이므로,')
print("     '수요감소 시 일반석 인상·프리미엄 할인'을 보편 법칙으로 해석하면 안 된다.")

 [A-4] 동일경기 쌍체 반사실 — 부상 OFF / ON
      date day_of_week  days_since_injury   cal      A   delta  shock_x  audience_off  audience_on   aud_%  revenue_off  revenue_on   rev_%
2024-07-03           수                2.0 -0.23 0.9602 -0.0414   0.9595          8688         8344 -3.9595    223586036   216881683 -2.9986
2024-07-15           월               14.0 -0.23 0.8243 -0.0355   0.9651          8688         8391 -3.4185    223586036   217817917 -2.5798
2024-07-31           수               30.0 -0.23 0.7686 -0.0331   0.9674          8688         8411 -3.1883    223586036   218203134 -2.4075
2024-08-15           목               45.0 -0.23 0.7551 -0.0325   0.9680          8688         8415 -3.1423    223586036   218296638 -2.3657
2024-08-28           수               58.0 -0.23 0.7516 -0.0324   0.9681          8688         8418 -3.1077    223586036   218320324 -2.3551

  5경기 평균  기대관중 -3.36%   기대수익 -2.54%
  ※ cal 열이 OFF/ON 양쪽에 동일하게 들어가 상쇄되므로 이 값이 부상 순효과다.

 [A-4] 최적가격 — 부상 없음 -> 부상 (5경기 평균)
   

In [54]:
# ==============================================================================
# [A-5] [시나리오 2] 정책 총효과 분해 — 달력 vs 부상
# ==============================================================================
#  조직위 대조군(3~4월 주말)과 처치군(7~8월 주중)을 원자료 그대로 비교하면 달력 효과가
#  함께 들어간다. 부상을 끈 상태와 켠 상태를 각각 비교해 두 성분을 분리한다.
#
#  ratio_off = 처치군 수익(부상 OFF) / 대조군 수익 : 달력·편성 효과
#  ratio_inj = 같은 처치군 경기의 수익(ON) / 수익(OFF) : 부상 순효과
#  ratio_on  = 처치군 수익(부상 ON) / 대조군 수익 : 관측되는 정책 총효과
#  ratio_inj를 처치군 평균수익 ON/OFF의 비로 정의하면 ratio_on=ratio_off*ratio_inj는
#  같은 처치군 평균과 같은 대조군 평균을 나누는 항등식이다. 프리미엄 좌석용량이 실제로
#  결합돼 있어도 이 곱셈 분해 자체는 성립한다. 단, 좌석포화는 가격 방향 해석에 중요하다.
# ==============================================================================
def group_ratio(injury_on):
    """시나리오 2 원자료 전체를 실행해 처치군/대조군 평균수익 비율을 반환한다."""
    g = df_games_scenarios[df_games_scenarios['scenario'].str.startswith('2.')].copy()
    if not injury_on:
        g['is_key_injury'] = 0
    r = run_evaluation_pipeline(g, M0_df, CAPACITY_df,
                                indicator_fn=indicator, verbose_clip=False)
    r['pair'] = g['pair'].to_numpy()
    m = r.groupby('pair')[['audience', 'revenue']].mean()
    return m, m.loc['처치군', 'revenue'] / m.loc['대조군', 'revenue']


m_on, ratio_on = group_ratio(True)
m_off, ratio_off = group_ratio(False)
# [A-4] 동일경기 쌍체에서 처치군 평균수익 ON/OFF 비를 직접 계산한 부상 순효과다.
ratio_inj = res2['revenue_on'].mean() / res2['revenue_off'].mean()

print('=' * 92)
print(' [A-5] 조직위 대조군 vs 처치군 — 정책 총효과 분해')
print('=' * 92)
print(m_on.round(0).to_string())
print(f"\n  총 격차       (달력 + 부상)  {ratio_on:.4f}  ->  {(ratio_on-1)*100:+.1f}%")
print(f"  ÷ 달력만      (부상 OFF)     {ratio_off:.4f}  ->  {(ratio_off-1)*100:+.1f}%")
print(f"  = 부상 순효과 (쌍체, [A-4])  {ratio_inj:.4f}  ->  {(ratio_inj-1)*100:+.1f}%")
print(f"\n  곱셈 분해 검산: {ratio_off:.4f} x {ratio_inj:.4f} = {ratio_off*ratio_inj:.4f}"
      f"   vs 총 격차 {ratio_on:.4f}   (오차 {abs(ratio_off*ratio_inj-ratio_on):.5f})")
print('  → 같은 처치군·대조군 평균을 사용한 비율 항등식이므로 정확히 곱으로 성립한다.')
injury_relative_pct = (ratio_inj - 1) * 100
injury_total_gap_pp = (ratio_on - ratio_off) * 100
print(f"\n  총 격차의 대부분은 달력이다. 부상 순효과는 같은 처치군 안에서"
      f" {injury_relative_pct:+.1f}%이고,")
print(f"  원 대조군 대비 전체 격차에 추가된 절대 기여분은 {injury_total_gap_pp:+.1f}%p다.")
print('  나머지는 처치군이 평일·혹서기에 편성된 결과다. 오히려 이 편성 자체의 가격·')
print('  프로모션 설계가 부상 대응보다 금액 임팩트가 크다.')

 [A-5] 조직위 대조군 vs 처치군 — 정책 총효과 분해
      audience      revenue
pair                       
대조군    13523.0  311542620.0
처치군     8396.0  217903939.0

  총 격차       (달력 + 부상)  0.6994  ->  -30.1%
  ÷ 달력만      (부상 OFF)     0.7177  ->  -28.2%
  = 부상 순효과 (쌍체, [A-4])  0.9746  ->  -2.5%

  곱셈 분해 검산: 0.7177 x 0.9746 = 0.6994   vs 총 격차 0.6994   (오차 0.00000)
  → 같은 처치군·대조군 평균을 사용한 비율 항등식이므로 정확히 곱으로 성립한다.

  총 격차의 대부분은 달력이다. 부상 순효과는 같은 처치군 안에서 -2.5%이고,
  원 대조군 대비 전체 격차에 추가된 절대 기여분은 -1.8%p다.
  나머지는 처치군이 평일·혹서기에 편성된 결과다. 오히려 이 편성 자체의 가격·
  프로모션 설계가 부상 대응보다 금액 임팩트가 크다.


In [55]:
# ==============================================================================
# [A-6] [시나리오 2] 재가격의 가치 — 가격 동결 vs 즉시 재최적화
# ==============================================================================
#  목적은 부상 자체의 손실과 가격을 다시 최적화해 회수할 수 있는 몫을 구분하는 것이다.
#
#  경기별 세 상태
#   ① rev_base: 부상 전 시장규모 M_off에서 부상 전 최적가격 x_off를 적용한 기준수익
#   ② rev_fix : 부상 후 시장규모 M_on에 기존 가격 x_off를 그대로 둔 가격동결 수익
#   ③ rev_opt : 부상 후 시장규모 M_on에 새 최적가격 x_on을 적용한 재최적화 수익
#
#  회수액   = rev_opt - rev_fix
#  회수_%p  = (rev_opt - rev_fix) / rev_base x 100
#  이 값은 부상 전 수익으로 완전히 복구했다는 뜻이 아니라, 동결했을 때의 추가 손실 중
#  가격 재조정만으로 줄인 부분이다. 고정 최적화 엔진은 수정하지 않고 두 번 호출한다.
scenario2_reprice_rows = []
for _, r in res2.iterrows():
    M_off = _M0 * np.exp(r['cal'])                 # 부상 없음: 공통 달력효과만 반영
    M_on = M_off * np.exp(r['delta'])              # 부상 발생: 부상 수요배수를 추가 적용
    x_off = optimize_single_game(M_off, _CAP).x    # 부상 전 최적가격 벡터
    x_on = optimize_single_game(M_on, _CAP).x      # 부상 후 새 최적가격 벡터

    rev_base = revenue_detail(x_off, M_off, _CAP)['revenue']
    rev_fix = revenue_detail(x_off, M_on, _CAP)['revenue']    # 수요만 감소, 가격은 동결
    rev_opt = revenue_detail(x_on, M_on, _CAP)['revenue']     # 감소한 수요에 맞춰 재최적화
    scenario2_reprice_rows.append({'date': r['date'], 'days': r['days_since_injury'],
                 '동결_%': (rev_fix / rev_base - 1) * 100,
                 '재최적화_%': (rev_opt / rev_base - 1) * 100,
                 '회수_%p': (rev_opt - rev_fix) / rev_base * 100,
                 '회수액_원': rev_opt - rev_fix})

reprice = pd.DataFrame(scenario2_reprice_rows)
print('=' * 100)
print(' [A-6] 재가격의 가치')
print('=' * 100)
print(reprice.round(3).to_string(index=False))
print(f"\n  평균   동결 {reprice['동결_%'].mean():+.2f}%  ->  재최적화"
      f" {reprice['재최적화_%'].mean():+.2f}%  =  회수 {reprice['회수_%p'].mean():+.2f}%p")
print(f"  5경기 합계 회수액 {reprice['회수액_원'].sum():,.0f}원")
print("  → '부상은 손실'보다 '손실의 일부는 즉시 재가격만으로 회수된다'가 제안서에 적합하다.")

 [A-6] 재가격의 가치
      date  days   동결_%  재최적화_%  회수_%p       회수액_원
2024-07-03   2.0 -4.196  -2.999  1.197 2676746.164
2024-07-15  14.0 -3.613  -2.580  1.033 2310313.187
2024-07-31  30.0 -3.373  -2.408  0.966 2158837.034
2024-08-15  45.0 -3.315  -2.366  0.949 2122007.147
2024-08-28  58.0 -3.300  -2.355  0.945 2112672.588

  평균   동결 -3.56%  ->  재최적화 -2.54%  =  회수 +1.02%p
  5경기 합계 회수액 11,380,576원
  → '부상은 손실'보다 '손실의 일부는 즉시 재가격만으로 회수된다'가 제안서에 적합하다.


In [56]:
# ==============================================================================
# [A-7] [시나리오 2] 민감도 — 공개 범위 OAT + rho×반감기 격자
# ==============================================================================
#  기준안 결과(res2)는 가격 제안용이고 sens2/decay_grid는 가정 검증용이다.
#  OAT는 한 번에 한 파라미터만 바꿔 결과 변동의 원인을 분리한다. 다만 A(d)의 rho와 h는
#  식 안에서 상호작용하므로 두 값은 3×3 격자도 추가해 조합 변화까지 확인한다.
#
#  [범위 근거]
#   · 주전 WAR 4/5/6: FanGraphs "What Is WAR?"의 All-Star/Superstar/MVP 경계.
#     https://library.fangraphs.com/misc/war/
#   · 백업 WAR 1: FanGraphs가 설명하는 평균 벤치선수 0~1 WAR 범위의 상단.
#     https://library.fangraphs.com/misc/war/replacement-level/
#   · beta 하·상단: Jeong & Nam(2021)의 beta=1.241, t=6.04에서 SE=beta/t를 역산해
#     beta±1.96SE로 만든 근사 95% 구간. 원 논문이 표준오차를 직접 싣지 않아 "근사"다.
#     https://www.abacademies.org/articles/the-habitual-consumption-propensity-of-korean-professional-baseball-fans-for-sustainable-management-12628.html
#   · rho={0.50,0.75,1.00}: rho의 직접 실증 추정치가 없으므로 강한 감쇠/기준/무감쇠를
#     공개하는 구조 민감도다. 0.75를 사실값으로 포장하지 않는 것이 목적이다.
#   · h={6.13,8,14}: 6.13일은 Thompson et al.(2021)의 NBA 부상 검색관심 유의기간 평균,
#     8일은 Prieto Curiel et al.(2019)이 추정한 일반 사건보도의 지수 반감기,
#     14일은 기존안의 느린 감쇠 스트레스다. 6.13일은 반감기 직접 추정치가 아니므로
#     하단 비교축일 뿐이며 기준값은 지수 반감기를 직접 보고한 8일이다.
#     https://doi.org/10.1515/jom-2020-0269
#     https://doi.org/10.1016/j.compenvurbsys.2019.02.004
#
#  win_rate는 부상 후 승률의 현실성 검산에만 쓰고 식(2-2)~(2-4)에 직접 들어가지 않으므로
#  결과가 변하지 않는 가짜 민감도 축으로 제시하지 않는다.
KBO_BETA_SE = INJURY['attendance_beta'] / 6.04
KBO_BETA_CI95 = (INJURY['attendance_beta'] - 1.96 * KBO_BETA_SE,
                 INJURY['attendance_beta'] + 1.96 * KBO_BETA_SE)

SCENARIO2_SENSITIVITY_CASES = [
    ('주전 WAR 4.0 (All-Star)', {'starter_war': 4.0}),
    ('기준: 주전5/백업0/rho.75/h8', {}),
    ('주전 WAR 6.0 (MVP 경계)', {'starter_war': 6.0}),
    ('백업 WAR 1.0 (벤치 상단)', {'backup_war': 1.0}),
    ('beta 근사95% 하단', {'attendance_beta': KBO_BETA_CI95[0]}),
    ('beta 근사95% 상단', {'attendance_beta': KBO_BETA_CI95[1]}),
    ('지속하한 rho=0.50', {'persistent_floor': 0.50}),
    ('무감쇠 rho=1.00', {'persistent_floor': 1.00}),
    ('빠른 관심창 h=6.13', {'attention_half_life_days': 6.13}),
    ('느린 감쇠 h=14', {'attention_half_life_days': 14.0}),
]

scenario2_sensitivity_rows = []
for case_label, overrides in SCENARIO2_SENSITIVITY_CASES:
    case_params = {**INJURY, **overrides}
    case_result = run_scenario2(case_params)
    scenario2_sensitivity_rows.append({
        'case': case_label,
        'rho': case_params['persistent_floor'],
        'half_life': case_params['attention_half_life_days'],
        'delta_min': case_result['delta'].min(),
        'delta_max': case_result['delta'].max(),
        'shock_x': case_result['shock_x'].mean(),
        'aud_%': case_result['aud_%'].mean(),
        'rev_%': case_result['rev_%'].mean(),
        'clipped': int(case_result['clipped'].sum()),
        'corner': int(case_result['corner'].sum()),
        'converged': bool(case_result['success'].all()),
    })

sens2 = pd.DataFrame(scenario2_sensitivity_rows)
sens2['usable'] = (
    (sens2['clipped'] == 0) & (sens2['corner'] == 0) & sens2['converged'])
print('=' * 118)
print(f' [A-7] OAT 민감도 ({len(SCENARIO2_SENSITIVITY_CASES)}케이스, 쌍체 반사실 기준)')
print('=' * 118)
print(sens2.round(4).to_string(index=False))
sens2_usable = sens2[sens2['usable']]
print(f"\n  유효 OAT 기대수익 변화 범위 "
      f"{sens2_usable['rev_%'].min():+.2f}% ~ {sens2_usable['rev_%'].max():+.2f}%")

# rho와 h는 상호작용하므로 OAT에 더해 모든 3×3 조합을 확인한다.
scenario2_decay_grid_rows = []
for rho in [0.50, 0.75, 1.00]:
    for half_life in [6.13, 8.0, 14.0]:
        case_params = {**INJURY, 'persistent_floor': rho,
                       'attention_half_life_days': half_life}
        case_result = run_scenario2(case_params)
        scenario2_decay_grid_rows.append({
            'rho': rho, 'half_life': half_life,
            'A_d2': _injury_attenuation(2, case_params),
            'A_d58': _injury_attenuation(58, case_params),
            'aud_%': case_result['aud_%'].mean(),
            'rev_%': case_result['rev_%'].mean(),
            'clipped': int(case_result['clipped'].sum()),
            'corner': int(case_result['corner'].sum()),
            'converged': bool(case_result['success'].all()),
        })

decay_grid = pd.DataFrame(scenario2_decay_grid_rows)
decay_grid['usable'] = (
    (decay_grid['clipped'] == 0) & (decay_grid['corner'] == 0) &
    decay_grid['converged'])
print('\n' + '-' * 118)
print(' [A-7] rho×반감기 3×3 구조 민감도')
print('-' * 118)
print(decay_grid.round(4).to_string(index=False))
print(f"\n  격자 기대수익 변화 범위 "
      f"{decay_grid['rev_%'].min():+.2f}% ~ {decay_grid['rev_%'].max():+.2f}%")
print('  → rho가 클수록 결장 지속효과가 커지고, 같은 rho에서는 h가 길수록 초기충격이')
print('    천천히 줄어 손실이 커진다. rho=1이면 감쇠 부분이 0이라 h가 결과를 바꾸지 않는다.')

assert (sens2['aud_%'] < 0).all() and (sens2['rev_%'] < 0).all()
assert (decay_grid['aud_%'] < 0).all() and (decay_grid['rev_%'] < 0).all()
assert sens2['usable'].all() and decay_grid['usable'].all()
print('  [OK] OAT·격자 전 케이스 관중/수익 감소, clip 0, corner 0, SLSQP 전건 수렴')


 [A-7] OAT 민감도 (10케이스, 쌍체 반사실 기준)
                 case  rho  half_life  delta_min  delta_max  shock_x   aud_%   rev_%  clipped  corner  converged  usable
주전 WAR 4.0 (All-Star) 0.75       8.00    -0.0331    -0.0259   0.9724 -2.7049 -2.0385        0       0       True    True
기준: 주전5/백업0/rho.75/h8 0.75       8.00    -0.0414    -0.0324   0.9656 -3.3633 -2.5413        0       0       True    True
  주전 WAR 6.0 (MVP 경계) 0.75       8.00    -0.0497    -0.0389   0.9589 -4.0239 -3.0416        0       0       True    True
   백업 WAR 1.0 (벤치 상단) 0.75       8.00    -0.0331    -0.0259   0.9724 -2.7049 -2.0385        0       0       True    True
        beta 근사95% 하단 0.75       8.00    -0.0279    -0.0219   0.9766 -2.2836 -1.7241        0       0       True    True
        beta 근사95% 상단 0.75       8.00    -0.0548    -0.0429   0.9547 -4.4360 -3.3516        0       0       True    True
        지속하한 rho=0.50 0.50       8.00    -0.0397    -0.0217   0.9735 -2.5967 -1.9578        0       0       True    Tru

In [57]:
# ==============================================================================
# [A-8] [시나리오 2] WAR 2축 민감도 — 스타 의존도 vs 뎁스
# ==============================================================================
#  식(2-2)를 정리하면 Delta_W=-(starter_war-backup_war)/144이므로 최종 delta는
#  두 WAR의 절대수준이 아니라 WAR_gap에만 의존한다. ([A-10] V8)
#
#  축 1 '스타 의존도': backup_war=0으로 고정하고 starter_war=4/5/6을 비교한다.
#                       스타와 대체선수 격차가 큰 구단일수록 부상 손실이 커지는지 본다.
#  축 2 '뎁스'       : starter_war=5로 고정하고 backup_war=0/1을 비교한다.
#                       구단이 통제 가능한 백업 전력 투자가 손실을 얼마나 줄이는지 본다.
#  4/5/6은 FanGraphs의 All-Star/Superstar/MVP 경계, 0/1은 replacement~평균 벤치
#  구간에서 가져왔다. 따라서 이 표에도 별도의 임의 WAR 값을 추가하지 않는다.
#  두 축을 전부 조합한 격자표는 같은 WAR_gap을 중복 표시하므로 만들지 않는다.
#  주전 WAR은 단기간에 바꾸기 어렵지만 백업 WAR은 로스터·육성 투자로 통제 가능하다.
# ==============================================================================
scenario2_war_axis_rows = []
for s_war in [4.0, 5.0, 6.0]:                       # 축 1: 공개 WAR 등급 경계 (백업 고정)
    r = run_scenario2({**INJURY, 'starter_war': s_war, 'backup_war': 0.0})
    scenario2_war_axis_rows.append({'축': '스타 의존도', '주전': s_war, '백업': 0.0, 'gap': s_war,
                 'delta(d=2)': r['delta'].min(), 'rev_%': r['rev_%'].mean()})
for b_war in [0.0, 1.0]:                            # 축 2: replacement~평균 벤치 구간
    r = run_scenario2({**INJURY, 'starter_war': 5.0, 'backup_war': b_war})
    scenario2_war_axis_rows.append({'축': '뎁스', '주전': 5.0, '백업': b_war, 'gap': 5.0 - b_war,
                 'delta(d=2)': r['delta'].min(), 'rev_%': r['rev_%'].mean()})

war_axes = pd.DataFrame(scenario2_war_axis_rows)
print('=' * 92)
print(' [A-8] WAR 2축 민감도')
print('=' * 92)
print(war_axes.round(4).to_string(index=False))
_b0 = war_axes.query("축=='뎁스' and 백업==0.0")['rev_%'].values[0]
_b1 = war_axes.query("축=='뎁스' and 백업==1.0")['rev_%'].values[0]
print(f"\n  뎁스 축: 백업을 replacement(0.0) -> 평균 벤치 상단(1.0 WAR)으로 높이면")
print(f"    부상 시 기대수익 손실이 {_b0:+.2f}% -> {_b1:+.2f}% 로"
      f" {(1 - _b1/_b0)*100:.0f}% 축소된다.")

 [A-8] WAR 2축 민감도
     축  주전  백업  gap  delta(d=2)   rev_%
스타 의존도 4.0 0.0  4.0     -0.0331 -2.0385
스타 의존도 5.0 0.0  5.0     -0.0414 -2.5413
스타 의존도 6.0 0.0  6.0     -0.0497 -3.0416
    뎁스 5.0 0.0  5.0     -0.0414 -2.5413
    뎁스 5.0 1.0  4.0     -0.0331 -2.0385

  뎁스 축: 백업을 replacement(0.0) -> 평균 벤치 상단(1.0 WAR)으로 높이면
    부상 시 기대수익 손실이 -2.54% -> -2.04% 로 20% 축소된다.


In [58]:
# ==============================================================================
# [A-9] [시나리오 2] 대안 사양 비교 — KBO 실증 선형식(본안) vs 로그비율식
# ==============================================================================
#  두 식 모두 동일한 KBO beta=1.241, WAR, A(d)를 사용한다. 따라서 결과 차이는
#  승률변화를 절대차로 쓸지 로그비율로 쓸지에만 생기며 시간감쇠 차이는 섞이지 않는다.
#
#  본안: delta_linear = beta x Delta_W
#        Jeong & Nam(2021)의 Delta ln(관중)=beta x Delta 승률 사양을 그대로 구현한다.
#
#  대안: eta_equiv = beta x W
#        delta_log = eta_equiv x ln((W+Delta_W)/W) x A(d)
#        ln(1+x) 약사를 통해 본안과 같은 국소 기울기를 갖는 로그비율 표현이다.
#
#  beta의 단위는 승률 1.0(=100%p) 변화당 로그관중 변화다. 1%p 승률 하락이면 delta는
#  -1.241x0.01=-0.01241이다. 1%p를 1.0으로 넣으면 100배 오류가 나므로 주의한다.
#  본안을 채택하는 이유는 더 큰 효과를 만들기 때문이 아니라 출처의 회귀식과 정확히
#  일치하기 때문이다. 로그비율식은 함수형 선택에 따른 차이만 확인하는 강건성 검증이다.
# ==============================================================================
def delta_injury_log_ratio(row, p=INJURY):
    """같은 KBO beta를 상대 승률 로그변화로 옮긴 비교용 대안; 기준 정책에는 미사용."""
    d = row.get('days_since_injury')
    if int(row.get('is_key_injury', 0) or 0) != 1 or d is None or pd.isna(d):
        return 0.0
    d = float(d)
    # 본안과 동일하게 식(2-6)의 적용구간 0<=d<days_miss를 사용한다.
    if not np.isfinite(d) or d < 0.0 or d >= p['days_miss']:
        return 0.0
    delta_w = -(p['starter_war'] - p['backup_war']) / p['season_games']
    win_after = p['win_rate'] + delta_w
    eta_equiv = p['attendance_beta'] * p['win_rate']
    A = _injury_attenuation(d, p)
    return eta_equiv * np.log(win_after / p['win_rate']) * A


def compare_spec(inj_fn):
    """달력을 고정한 쌍체 비교로 부상 사양의 delta 범위와 평균 수익효과를 반환한다."""
    out, treat = paired_counterfactual('2.', 'is_key_injury',
                                       lambda r: delta_calendar(r) + inj_fn(r))
    d = treat.apply(inj_fn, axis=1)
    return d.min(), d.max(), out['rev_%'].mean()


# 사양 비교 결과도 별도 이름에 저장해 앞선 민감도 표의 행 목록을 덮어쓰지 않는다.
scenario2_spec_rows = []
for s_war in [4.0, 5.0, 6.0]:
    _p_spec = {**INJURY, 'starter_war': s_war}
    lo, hi, rev = compare_spec(lambda r, p=_p_spec: delta_injury(r, p))
    scenario2_spec_rows.append({'사양': f'본안 KBO 선형  주전 {s_war}',
                                'delta_min': lo, 'delta_max': hi, 'rev_%': rev})
    lo, hi, rev = compare_spec(lambda r, p=_p_spec: delta_injury_log_ratio(r, p))
    scenario2_spec_rows.append({'사양': f'대안 로그비율  주전 {s_war}',
                                'delta_min': lo, 'delta_max': hi, 'rev_%': rev})

print('=' * 96)
print(' [A-9] 사양 비교')
print('=' * 96)
_spec_df = pd.DataFrame(scenario2_spec_rows)
print(_spec_df.round(4).to_string(index=False))

_eta_equiv = INJURY['attendance_beta'] * INJURY['win_rate']
print(f"\n  동일기울기 환산: eta_equiv = beta x W = {INJURY['attendance_beta']:.3f}"
      f" x {INJURY['win_rate']:.4f} = {_eta_equiv:.4f}")
print('  → 두 사양 모두 기대수익 감소로 결론이 같고, 본안은 KBO 추정식과 정확히 일치한다.')

 [A-9] 사양 비교
               사양  delta_min  delta_max   rev_%
본안 KBO 선형  주전 4.0    -0.0331    -0.0259 -2.0385
  대안 로그비율  주전 4.0    -0.0340    -0.0266 -2.0912
본안 KBO 선형  주전 5.0    -0.0414    -0.0324 -2.5413
  대안 로그비율  주전 5.0    -0.0427    -0.0335 -2.6241
본안 KBO 선형  주전 6.0    -0.0497    -0.0389 -3.0416
  대안 로그비율  주전 6.0    -0.0516    -0.0404 -3.1611

  동일기울기 환산: eta_equiv = beta x W = 1.241 x 0.5493 = 0.6817
  → 두 사양 모두 기대수익 감소로 결론이 같고, 본안은 KBO 추정식과 정확히 일치한다.


In [59]:
# ==============================================================================
# [A-10] [시나리오 2] 검증 V0~V9 · 결과 원인 · 구조 한계
# ==============================================================================
#  V0 : WAR·승률·경기수·beta·rho·반감기의 범위와 산출 관계가 정상인가
#  V1 : 함수 결과가 식(2-1)~(2-6)의 독립 재계산과 모든 경과일에서 일치하는가
#  V2 : 동일경기 OFF/ON에서 달력 성분이 정확히 상쇄되는가
#  V3 : 총 log_shock이 고정 SHOCK_CLIP에 잘리지 않는가
#  V4 : 최적가격이 0.7×/1.5× 경계에 붙지 않는가
#  V5 : OFF/ON의 SLSQP가 모든 경기에서 정상 수렴하는가
#  V6 : 좌석등급별 소진율로 일반석 인상·프리미엄 인하의 원인을 확인했는가
#  V7 : 공개범위 OAT와 rho×h 격자에서 관중·수익 감소 부호가 유지되는가
#  V8 : 같은 WAR_gap·같은 d는 같은 delta를 내는가
#  V9 : A(d)는 d에 따라 감소하고 rho 아래로 내려가지 않으며 d=62에서만 게이트되는가
# ==============================================================================
p = INJURY
assert p['starter_war'] >= p['backup_war'] >= 0
assert 0 < p['win_rate'] <= 1 and p['attendance_beta'] > 0
assert p['regular_season_days'] > 0 and p['games_miss'] > 0 and p['days_miss'] > 0
assert 0 <= p['persistent_floor'] <= 1 and p['attention_half_life_days'] > 0
assert np.isclose(p['games_miss'], p['season_games'] * p['days_miss'] /
                  p['regular_season_days'])
_gap0 = p['starter_war'] - p['backup_war']
assert p['win_rate'] - _gap0 / p['season_games'] > 0
print('  [OK] V0  파라미터 범위·부호·산출관계 정상')

# V1: delta_injury를 다시 부르지 않고 식을 밖에서 재계산해 구현 오타를 잡는다.
_delta_w = -_gap0 / p['season_games']
for _, x in res2.iterrows():
    _A_expected = (p['persistent_floor'] +
                   (1 - p['persistent_floor']) *
                   2 ** (-x['days_since_injury'] / p['attention_half_life_days']))
    _delta_expected = p['attendance_beta'] * _delta_w * _A_expected
    assert np.isclose(x['A'], _A_expected)
    assert np.isclose(x['delta'], _delta_expected)
_d61_A = (p['persistent_floor'] + (1 - p['persistent_floor']) *
          2 ** (-61 / p['attention_half_life_days']))
_d61_expected = p['attendance_beta'] * _delta_w * _d61_A
assert np.isclose(delta_injury({'is_key_injury': 1, 'days_since_injury': 61}),
                  _d61_expected)
assert delta_injury({'is_key_injury': 1, 'days_since_injury': 62}) == 0.0
print(f'  [OK] V1  식 독립 재계산 일치; d=61 delta={_d61_expected:+.6f}, d=62 복귀')

_g2 = df_games_scenarios[df_games_scenarios['scenario'].str.startswith('2.')]
_tr = _g2[_g2['pair'] == '처치군'].reset_index(drop=True)
_offd = _tr.copy()
_offd['is_key_injury'] = 0
for i in range(len(_tr)):
    assert np.isclose(indicator(_tr.iloc[i]) - indicator(_offd.iloc[i]),
                      delta_injury(_tr.iloc[i]))
print('  [OK] V2  OFF/ON 차이=부상 delta; 날짜·요일·월 성분 완전 상쇄')

assert res2['clipped'].sum() == 0
_tot = res2['cal'] + res2['delta']
print(f'  [OK] V3  SHOCK_CLIP{SHOCK_CLIP} 발동 0건'
      f'  (총 log_shock {_tot.min():.4f}~{_tot.max():.4f})')

assert res2['corner'].sum() == 0
print('  [OK] V4  가격 경계 접촉 0건 — 효과가 고정 가격범위에 잘린 결과가 아님')

assert res2['success'].all()
print('  [OK] V5  SLSQP OFF/ON 전건 수렴')

# V6: 전체 관중/총좌석만 보면 P석 포화를 숨기므로 G/P를 분리한다.
_fill_rows = []
for _, x in res2.iterrows():
    for state in ['off', 'on']:
        _M = _M0 * np.exp(x['cal'] + (x['delta'] if state == 'on' else 0.0))
        _prices = np.array([x[f'{c}_{state}'] for c in
                            ['p_D7_G', 'p_D7_P', 'p_D3_G', 'p_D3_P']])
        _det = revenue_detail(_prices, _M, _CAP)
        _fill_rows.append({
            'state': state,
            'G': (_det['Q_D7_G'] + _det['Q_D3_G']) / _det['cap_G'],
            'P': (_det['Q_D7_P'] + _det['Q_D3_P']) / _det['cap_P']})
_fills = pd.DataFrame(_fill_rows)
assert _fills[['G', 'P']].max().max() <= 1 + 1e-9
assert np.isclose(_fills['P'].max(), 1.0, atol=1e-6)
print(f"  [진단] V6  최대 소진율 G {_fills['G'].max()*100:.1f}% /"
      f" P {_fills['P'].max()*100:.1f}% — P 용량제약 결합")
print('              P석은 이미 희소하므로 수요 감소 후 P 가격을 내려 물량을 지키고,')
print('              MNL 교차대체를 통해 G 가격을 올려 P의 상대효용을 높인다.')

assert (sens2['rev_%'] < 0).all() and (sens2['aud_%'] < 0).all()
assert (decay_grid['rev_%'] < 0).all() and (decay_grid['aud_%'] < 0).all()
assert sens2['usable'].all() and decay_grid['usable'].all()
print(f'  [OK] V7  OAT {len(sens2)}개 + rho×h 격자 {len(decay_grid)}개 모두'
      ' 관중/수익 감소, clip/corner 0, 수렴 성공')

_row = {'is_key_injury': 1, 'days_since_injury': 14}
_iso = [(5.0, 0.0), (6.0, 1.0), (7.0, 2.0)]
_v = [delta_injury(_row, {**INJURY, 'starter_war': s, 'backup_war': b})
      for s, b in _iso]
assert np.allclose(_v, _v[0])
print(f"  [OK] V8  같은 WAR_gap=5, d=14인 {', '.join(f'({s},{b})' for s,b in _iso)}"
      f" 모두 delta={_v[0]:+.6f}")

_days = np.arange(0, p['days_miss'])
_avec = np.array([_injury_attenuation(d, p) for d in _days])
assert np.isclose(_avec[0], 1.0)
assert np.all(np.diff(_avec) <= 1e-12)
assert (_avec >= p['persistent_floor'] - 1e-12).all()
assert delta_injury({'is_key_injury': 1, 'days_since_injury': -1}) == 0.0
assert delta_injury({'is_key_injury': 1, 'days_since_injury': 62}) == 0.0
print(f"  [OK] V9  A(0)=1, 단조감소, 최소 {_avec.min():.4f}>=rho"
      f" {p['persistent_floor']}; d<0 부상 전 OFF / d=62 복귀 OFF")

print('\n  [결과 원인 1] 경과일이 길수록 손실이 작아지는 것은 초기 관심 부분만 감쇠하기 때문이다.')
print('    선수의 실제 결장 전력은 rho 하한으로 남는다. 이는 회복확률 모형이 아니다.')
print('  [결과 원인 2] 관중보다 수익 감소가 작은 것은 부상 ON 상태에서 가격을 다시 최적화해')
print('    동결가격 대비 손실 일부를 회수하기 때문이다. [A-6]에서 금액을 별도 계산한다.')
print('  [결과 원인 3] 조직위 원 대조군과의 큰 격차는 부상뿐 아니라 평일·혹서기 달력구성이')
print('    섞인 총효과다. 부상 순효과는 [A-4] 동일경기 쌍체 값만 인용한다.')
print('  [구조 한계] 고정 baseline은 한 경기의 M_t를 D-7과 D-3에 공통 적용한다.')
print('    7/3 경기는 부상일 7/1 이전의 D-7 가격에도 충격이 소급된다. 고정함수를 바꾸지 않고')
print('    보고서에서 D-7은 모형상 반사실 가격, 실무 적용은 공지 후 첫 재가격(D-3)으로 명시한다.')


  [OK] V0  파라미터 범위·부호·산출관계 정상
  [OK] V1  식 독립 재계산 일치; d=61 delta=-0.032372, d=62 복귀
  [OK] V2  OFF/ON 차이=부상 delta; 날짜·요일·월 성분 완전 상쇄
  [OK] V3  SHOCK_CLIP(-0.6, 0.8) 발동 0건  (총 log_shock -0.2714~-0.2624)
  [OK] V4  가격 경계 접촉 0건 — 효과가 고정 가격범위에 잘린 결과가 아님
  [OK] V5  SLSQP OFF/ON 전건 수렴
  [진단] V6  최대 소진율 G 25.4% / P 100.0% — P 용량제약 결합
              P석은 이미 희소하므로 수요 감소 후 P 가격을 내려 물량을 지키고,
              MNL 교차대체를 통해 G 가격을 올려 P의 상대효용을 높인다.
  [OK] V7  OAT 10개 + rho×h 격자 9개 모두 관중/수익 감소, clip/corner 0, 수렴 성공
  [OK] V8  같은 WAR_gap=5, d=14인 (5.0,0.0), (6.0,1.0), (7.0,2.0) 모두 delta=-0.035520
  [OK] V9  A(0)=1, 단조감소, 최소 0.7513>=rho 0.75; d<0 부상 전 OFF / d=62 복귀 OFF

  [결과 원인 1] 경과일이 길수록 손실이 작아지는 것은 초기 관심 부분만 감쇠하기 때문이다.
    선수의 실제 결장 전력은 rho 하한으로 남는다. 이는 회복확률 모형이 아니다.
  [결과 원인 2] 관중보다 수익 감소가 작은 것은 부상 ON 상태에서 가격을 다시 최적화해
    동결가격 대비 손실 일부를 회수하기 때문이다. [A-6]에서 금액을 별도 계산한다.
  [결과 원인 3] 조직위 원 대조군과의 큰 격차는 부상뿐 아니라 평일·혹서기 달력구성이
    섞인 총효과다. 부상 순효과는 [A-4] 동일경기 쌍체 값만 인용한다.
  [구조 한계] 고정 baseline은 한 경기의 M_t를 D-7과 D-3에

In [60]:
# ==============================================================================
# [A-11] [시나리오 3] 기습 장마 예보 — 로지스틱 충격 · 쌍체 반사실 검증
# ==============================================================================
#  ON은 조직위 실제 강수확률, OFF는 같은 경기의 평상시 기준확률 0.05다.
#  날짜·요일·월은 동일하므로 달력 성분이 상쇄되고 차이는 강수예보 하나뿐이다.
#  아래 기준안 결과(res3)는 실제 가격 제안과 보고서 본문에 사용하는 값이다.
#  민감도 분석(sens3)은 RAIN 기준안을 바꾸지 않는 검증 절차로, 최대충격·기울기·
#  로지스틱 중심확률을 한 축씩 변경한다. K=0.30은 채택 후보가 아니라 가격 경계해가
#  얼마나 빨리 발생하는지 확인하는 스트레스 케이스다.
#
#  K는 [P-4]의 2단계 스크리닝으로 정했고, 아래 두 수치는 층위가 다르다.
#   · K=0.25: [2단계 실증] 실제 채택·운영값. 조직위 6경기에서 clip·가격 경계해 0건이다.
#   · K=0.40: [1단계 이론] 채택 후보가 아니라 delta_rain >= -0.4를 지키기 위한 방어선이다.
#             K가 0.4를 넘으면 강수확률이 높은 경기의 delta_rain이 -0.4보다 낮아질 수
#             있고, 음의 달력충격과 합쳐져 고정 SHOCK_CLIP 하한 -0.6에 붙는 코너
#             솔루션 위험이 커진다. 실제 가격 경계해는 그보다 작은 K에서도 먼저 생길
#             수 있으므로, 민감도 결과에서 안전한 K=0.25를 더 보수적으로 채택한다.
# ==============================================================================
def scenario3_indicator(row, p=RAIN):
    """시나리오 3 민감도용 지표 진입점. 다른 지표는 기준안으로 고정한다."""
    return (delta_calendar(row) + delta_star_record(row)
            + delta_injury(row) + delta_rain(row, p))


def run_scenario3(p=RAIN):
    fn = lambda row: scenario3_indicator(row, p)
    return paired_counterfactual(
        '3.', 'rain_prob', fn, off_value=p['normal_prob'])


s3 = df_games_scenarios[df_games_scenarios['scenario'].str.startswith('3.')].copy()
s3['cal'] = s3.apply(delta_calendar, axis=1)
s3['delta_rain'] = s3.apply(delta_rain, axis=1)
s3['rain_x'] = np.exp(s3['delta_rain'])
s3['total_delta'] = s3['cal'] + s3['delta_rain']
print('=' * 96)
print(' [A-11] 시나리오 3 데이터 구성 · 강수확률 · 로지스틱 충격')
print('=' * 96)
print(s3[['pair', 'date', 'day_of_week', 'rain_prob', 'cal', 'delta_rain',
          'rain_x', 'total_delta']].round(4).to_string(index=False))
print(f"\n  달력 격차 {s3[s3['pair']=='처치군']['cal'].mean() - s3[s3['pair']=='대조군']['cal'].mean():+.4f}"
      f"  (대조군 주말 4/6 vs 처치군 주말 1/6). 원자료 비교 불가.")

# --- 기준안의 SHOCK_CLIP 점검 -------------------------------------------------
s3t = s3[s3['pair'] == '처치군']
n_clip = int(((s3t['total_delta'] < SHOCK_CLIP[0]) |
              (s3t['total_delta'] > SHOCK_CLIP[1])).sum())
print(f'\n  [clip 점검] 총 log_shock {s3t["total_delta"].min():+.4f}'
      f' ~ {s3t["total_delta"].max():+.4f}  /  SHOCK_CLIP {SHOCK_CLIP}  /  clip {n_clip}건')

# --- 동일경기 쌍체 반사실: 실제 확률(ON) vs 평상시 5%(OFF) ----------------------
res3, _ = run_scenario3()
res3['rain_prob'] = s3t['rain_prob'].to_numpy()
res3['delta_rain'] = s3t['delta_rain'].to_numpy()
print('\n' + '-' * 96)
print(f' [A-11] 쌍체 반사실 — 평상시 {RAIN["normal_prob"]:.0%}(OFF) / 실제 확률(ON)')
print('-' * 96)
print(res3[['date', 'day_of_week', 'rain_prob', 'cal', 'delta_rain',
            'audience_off', 'audience_on', 'aud_%',
            'revenue_off', 'revenue_on', 'rev_%']].round(4).to_string(index=False))
print(f"\n  6경기 평균  기대관중 {res3['aud_%'].mean():+.2f}%   기대수익 {res3['rev_%'].mean():+.2f}%")
print(f"  검증: clip {int(res3['clipped'].sum())}건 / corner hit {int(res3['corner'].sum())}건"
      f" / SLSQP 전건 수렴 {bool(res3['success'].all())}")
print('  ※ 동일 경기의 달력 성분은 OFF/ON 양쪽에 같게 들어가므로 위 비율은 강수예보 순효과다.')

# --- 검증용 민감도 분석: 기준안과 분리하여 한 축씩 변경(OAT) --------------------
#  max_shock: 강수충격의 최대 크기, steepness: 중간확률대 반응 속도,
#  midpoint: 수요 감소가 절반 수준에 도달하는 중심 강수확률이다.
#  '기준안'의 빈 dict는 RAIN을 그대로 사용하고, 나머지는 표시된 한 축만 덮어쓴다.
SCENARIO3_SENSITIVITY_CASES = [
    ('최대충격 0.15', {'max_shock': 0.15}),
    ('기준안',        {}),
    ('최대충격 0.30', {'max_shock': 0.30}),  # 강한 스트레스: 경계해 여부 확인
    ('기울기 4',      {'steepness': 4.0}),
    ('기울기 8',      {'steepness': 8.0}),
    ('기준확률 0.4',  {'midpoint': 0.4}),
    ('기준확률 0.6',  {'midpoint': 0.6}),
]

#  RAIN은 실제 운영값으로 고정하고, 각 반복에서 복사본을 만들어 검증한다.
#  따라서 민감도 셀을 실행한 뒤에도 my_virtual_indicators는 계속 K=0.25를 사용한다.
scenario3_sensitivity_rows = []
for case_label, overrides in SCENARIO3_SENSITIVITY_CASES:
    case_params = {**RAIN, **overrides}
    case_result, _ = run_scenario3(case_params)
    case_deltas = s3t.apply(
        lambda row: delta_rain(row, case_params), axis=1)
    scenario3_sensitivity_rows.append({
        'case': case_label,
        'K': case_params['max_shock'],
        'steepness': case_params['steepness'],
        'midpoint': case_params['midpoint'],
        'normal_prob': case_params['normal_prob'],
        'delta_min': case_deltas.min(), 'delta_max': case_deltas.max(),
        'aud_%': case_result['aud_%'].mean(),
        'rev_%': case_result['rev_%'].mean(),
        'clipped': int(case_result['clipped'].sum()),
        'corner': int(case_result['corner'].sum()),
        'converged': bool(case_result['success'].all()),
    })

sens3 = pd.DataFrame(scenario3_sensitivity_rows)
#  usable은 보고서 정량 인용 가능 여부다. clip이 없더라도 가격이 0.7x/1.5x 경계에
#  붙으면 최적 가격이 경계에 의해 잘린 것이므로, 해당 케이스는 방향성 점검에만 쓰고
#  효과 크기의 최솟값·최댓값 계산에서는 제외한다.
sens3['usable'] = (
    (sens3['clipped'] == 0) & (sens3['corner'] == 0) & sens3['converged'])
print('\n' + '-' * 112)
print(f' [A-11] 시나리오 3 민감도 ({len(SCENARIO3_SENSITIVITY_CASES)}케이스, 쌍체 반사실 기준)')
print('-' * 112)
print(sens3.round(4).to_string(index=False))
sens3_usable = sens3[sens3['usable']]
print(f"\n  유효 케이스 기대수익 변화 범위 "
      f"{sens3_usable['rev_%'].min():+.2f}% ~ {sens3_usable['rev_%'].max():+.2f}%")

#  스트레스 케이스는 unusable이어도 부호 점검에는 포함한다. 반면 clip과 수렴 실패는
#  모형 비교 자체를 훼손하므로 어떤 케이스에서도 허용하지 않는다.
assert (sens3['aud_%'] < 0).all() and (sens3['rev_%'] < 0).all()
assert sens3['clipped'].sum() == 0 and sens3['converged'].all()
print('  [OK] 전 케이스 관중·수익 감소 / clip 0건 / SLSQP 전건 수렴')
if not sens3['usable'].all():
    bad = ', '.join(sens3.loc[~sens3['usable'], 'case'])
    print(f'  [주의] 가격 경계해 발생으로 정량 인용 제외: {bad}')

 [A-11] 시나리오 3 데이터 구성 · 강수확률 · 로지스틱 충격
pair       date day_of_week  rain_prob   cal  delta_rain  rain_x  total_delta
 대조군 2024-04-19           금       0.05  0.10     -0.0157  0.9844       0.0843
 대조군 2024-04-20           토       0.05  0.10     -0.0157  0.9844       0.0843
 대조군 2024-04-21           일       0.05  0.10     -0.0157  0.9844       0.0843
 대조군 2024-04-26           금       0.05  0.10     -0.0157  0.9844       0.0843
 대조군 2024-04-27           토       0.05  0.10     -0.0157  0.9844       0.0843
 대조군 2024-04-28           일       0.05  0.10     -0.0157  0.9844       0.0843
 처치군 2024-09-02           월       0.85 -0.17     -0.2227  0.8003      -0.3927
 처치군 2024-09-03           화       0.75 -0.17     -0.2044  0.8151      -0.3744
 처치군 2024-09-04           수       0.65 -0.17     -0.1777  0.8372      -0.3477
 처치군 2024-09-08           일       0.50  0.24     -0.1250  0.8825       0.1150
 처치군 2024-09-09           월       0.35 -0.17     -0.0723  0.9303      -0.2423
 처치군 2024-09-10          

In [61]:
# ==============================================================================
# [A-12] [전 시나리오 공통] 최종 가격 제안표 · 이벤트 순효과 · 표준경기 정규화 비교
# ==============================================================================
#  이 셀은 앞의 모든 분석을 보고서에 그대로 옮길 수 있는 3개의 표로 정리한다.
#
#  [표 1] 최종 가격 제안표
#    조직위 30경기 전부에 대해 고정 SLSQP 엔진이 낸 D-7/D-3 × 일반/프리미엄 4개 가격을
#    원화와 기본가 배수로 함께 싣는다. log_shock을 cal(달력) + event(시나리오 지표)로
#    분해해 각 가격이 어느 성분에서 나왔는지 한 줄에서 읽히게 한다.
#    clip/corner/수렴 3개 품질열이 모두 통과한 행만 정량 인용 대상이다.
#
#  [표 2] 이벤트 순효과 (보고서 헤드라인 수치)
#    [A-3]/[A-4]/[A-11]의 동일경기 쌍체 반사실 결과다. 같은 경기에서 이벤트 플래그만
#    바꾸므로 달력 성분이 정확히 상쇄된다. 보고서에서 "대기록/부상/장마의 효과"라고
#    부를 수 있는 값은 이 표뿐이다.
#
#  [표 3] 표준경기 정규화 비교  ★ 절대금액 비교는 반드시 이 표로만 한다
#    표 1의 절대수익은 시나리오 간 비교가 불가능하다. 세 시나리오 모두 홈팀·M0·구장이
#    같은데도 대조군 수익이 267/312/282백만원으로 갈리는데, 이 격차는 전부 편성 달력
#    (월·요일)에서 나온다. 그대로 비교하면 부상 처치군(218백만)이 장마 처치군(224백만)
#    보다 낮아 "부상이 장마보다 나쁘다"는 결론이 나오지만, 표 2의 순효과는 부상 -2.5%,
#    장마 -8.6%로 정반대다.
#    따라서 세 이벤트를 같은 축에 올리려면 달력조건을 하나로 고정해야 한다. 여기서는
#    성수기 주말(9월 토요일, delta_cal = +0.24)을 표준경기로 두고, 각 시나리오 처치군
#    경기의 이벤트 delta만 그 위에 얹어 OFF/ON을 각각 재최적화한다. 고정 수요모형·
#    희소성 함수·SLSQP는 그대로 사용한다.
# ==============================================================================
FINAL_CAL_REF_MONTH, FINAL_CAL_REF_DOW = 9, '토'      # 표준경기: 성수기 주말
CAL_REF = delta_calendar({'date': f'2024-{FINAL_CAL_REF_MONTH:02d}-07',
                          'day_of_week': FINAL_CAL_REF_DOW})

# ---- [표 1] 최종 가격 제안표 -------------------------------------------------
final_price_table = df_games_scenarios[['scenario', 'pair', 'date', 'day_of_week']].copy()
final_price_table['cal'] = df_games_scenarios.apply(delta_calendar, axis=1)
final_price_table['event'] = df_games_scenarios.apply(
    lambda r: indicator(r) - delta_calendar(r), axis=1)
for _c in ['log_shock', 'shock_x', 'M_t', 'audience', 'revenue', 'clipped',
           'corner_hits', 'success']:
    final_price_table[_c] = res_scenarios[_c].to_numpy()
for _c, _b in [('p_D7_G', bG), ('p_D7_P', bP), ('p_D3_G', bG), ('p_D3_P', bP)]:
    final_price_table[_c] = res_scenarios[_c].to_numpy()
    final_price_table[_c + '_x'] = (final_price_table[_c] / _b).round(2)
final_price_table['인용가능'] = (~final_price_table['clipped'].astype(bool)
                             & (final_price_table['corner_hits'] == 0)
                             & final_price_table['success'].astype(bool))

print('=' * 126)
print(f' [A-12] 표 1. 최종 가격 제안표 — 기본가 일반석 {bG:,}원 / 프리미엄석 {bP:,}원')
print('=' * 126)
print(final_price_table[['scenario', 'pair', 'date', 'day_of_week', 'cal', 'event',
                         'log_shock', 'M_t', 'p_D7_G', 'p_D7_G_x', 'p_D7_P', 'p_D7_P_x',
                         'p_D3_G', 'p_D3_G_x', 'p_D3_P', 'p_D3_P_x',
                         'audience', 'revenue', '인용가능']].to_string(index=False))
assert final_price_table['인용가능'].all()
print(f"\n  [OK] 30경기 전건 clip 0 / corner hit 0 / SLSQP 수렴 — 표 전체를 정량 인용할 수 있다.")
print('  ※ 가격은 네 경우 모두 1.0x를 넘는다. 기본가가 MNL 최적가보다 낮게 설정돼 있어')
print('    수요가 줄어든 경기에서도 인상이 최적이며, 조정은 인상폭의 크기로 나타난다.')

# ---- [표 2] 이벤트 순효과 (쌍체 반사실) --------------------------------------
event_net_effect = pd.DataFrame([
    {'시나리오': '1. 스타 대기록', '경기수': len(res1), '지표': 'delta_star',
     'delta_min': res1['delta_star'].min(), 'delta_max': res1['delta_star'].max(),
     '관중_%': res1['aud_%'].mean(), '수익_%': res1['rev_%'].mean(),
     '민감도범위_수익_%': f"{sens1_usable['rev_%'].min():+.2f} ~ {sens1_usable['rev_%'].max():+.2f}"},
    {'시나리오': '2. 핵심타자 부상', '경기수': len(res2), '지표': 'delta_injury',
     'delta_min': res2['delta'].min(), 'delta_max': res2['delta'].max(),
     '관중_%': res2['aud_%'].mean(), '수익_%': res2['rev_%'].mean(),
     '민감도범위_수익_%': f"{sens2_usable['rev_%'].min():+.2f} ~ {sens2_usable['rev_%'].max():+.2f}"},
    {'시나리오': '3. 기습장마 예보', '경기수': len(res3), '지표': 'delta_rain',
     'delta_min': res3['delta_rain'].min(), 'delta_max': res3['delta_rain'].max(),
     '관중_%': res3['aud_%'].mean(), '수익_%': res3['rev_%'].mean(),
     '민감도범위_수익_%': f"{sens3_usable['rev_%'].min():+.2f} ~ {sens3_usable['rev_%'].max():+.2f}"},
])
print('\n' + '=' * 126)
print(' [A-12] 표 2. 이벤트 순효과 — 동일경기 쌍체 반사실 (달력 성분 완전 상쇄)')
print('=' * 126)
print(event_net_effect.round(4).to_string(index=False))
print('  → 보고서에서 "이벤트의 효과"로 인용할 수 있는 수치는 이 표뿐이다.')

# ---- [표 3] 표준경기 정규화 비교 ---------------------------------------------
#  각 처치군 경기의 OFF/ON 이벤트 delta만 떼어내 표준경기 달력 위에 얹고 두 상태를
#  각각 재최적화한다. OFF 정의는 [A-3]/[A-4]/[A-11]의 쌍체 반사실과 동일하게
#  시나리오 1·2는 플래그 0, 시나리오 3은 평상시 강수확률 0.05로 맞춘다.
#  따라서 표 3의 차이_%는 표 2와 같은 이벤트를 같은 방식으로 잰 값이며, 두 표의 %가
#  다른 부분은 오직 '어느 달력 수준에서 쟀는가'에서 온다. (P석 포화로 수익함수가
#  M_t에 비선형이라 같은 delta라도 출발 수요 수준이 다르면 %가 달라진다.)
SCENARIO_OFF_SPEC = {                       # 시나리오 접두사 -> (이벤트 컬럼, OFF 값)
    '1.': ('is_star_record', 0),
    '2.': ('is_key_injury', 0),
    '3.': ('rain_prob', RAIN['normal_prob']),
}

_M_ref = _M0 * np.exp(CAL_REF)
_det_ref = revenue_detail(optimize_single_game(_M_ref, _CAP).x, _M_ref, _CAP)

def _standardized(delta_event):
    """표준경기 달력 위에 이벤트 delta만 얹어 재최적화한다."""
    M = _M0 * np.exp(CAL_REF + delta_event)
    return revenue_detail(optimize_single_game(M, _CAP).x, M, _CAP)

standardized_rows = []
_treat = df_games_scenarios[df_games_scenarios['pair'] == '처치군']
for _sc, _grp in _treat.groupby('scenario'):
    _col, _off_val = SCENARIO_OFF_SPEC[_sc[:2]]
    _grp_off = _grp.copy()
    _grp_off[_col] = _off_val
    _rev_on, _rev_off, _aud_on, _aud_off = [], [], [], []
    for _i in range(len(_grp)):
        _r_on, _r_off = _grp.iloc[_i], _grp_off.iloc[_i]
        _d_on = _standardized(indicator(_r_on) - delta_calendar(_r_on))
        _d_off = _standardized(indicator(_r_off) - delta_calendar(_r_off))
        _rev_on.append(_d_on['revenue']); _rev_off.append(_d_off['revenue'])
        _aud_on.append(_d_on['total_audience']); _aud_off.append(_d_off['total_audience'])
    standardized_rows.append({
        '시나리오': _sc, '경기수': len(_grp),
        '표준_OFF_수익': np.mean(_rev_off), '표준_ON_수익': np.mean(_rev_on),
        '차이_원': np.mean(_rev_on) - np.mean(_rev_off),
        '차이_%': (np.mean(_rev_on) / np.mean(_rev_off) - 1) * 100,
        '관중차이_명': np.mean(_aud_on) - np.mean(_aud_off),
    })
standardized = pd.DataFrame(standardized_rows)
standardized['표2_수익_%'] = event_net_effect['수익_%'].to_numpy()

print('\n' + '=' * 126)
print(f' [A-12] 표 3. 표준경기 정규화 비교 — 9월 토요일(delta_cal {CAL_REF:+.2f}) 고정')
print('=' * 126)
print(f"  표준경기(이벤트 없음): M_t {_M_ref:,.0f}명  기대관중 {_det_ref['total_audience']:,.0f}명"
      f"  기대수익 {_det_ref['revenue']:,.0f}원")
print(standardized.round(2).to_string(index=False))
print('\n  → 달력을 고정하면 손실 크기가 장마 > 부상 순으로 표 2와 같은 순서가 된다.')
print('    표 1의 원자료 절대수익에서는 부상 처치군(218백만) < 장마 처치군(224백만)이라')
print('    순서가 뒤집힌다. 이는 부상 처치군이 7~8월 주중, 장마 처치군이 9월(일요일 포함)에')
print('    편성된 결과일 뿐 이벤트 크기와 무관하다. 절대금액 비교는 반드시 이 표로만 한다.')
print('  → 표 3과 표 2의 % 격차는 P석 포화로 수익함수가 M_t에 비선형이기 때문이다.')
print('    쌍체 % 역시 "그 경기의 달력조건 하에서의 효과"이며 이벤트 고유 탄력성이 아니다.')

# ---- 절대 비교가 왜 위험한지 보여주는 단일 반례 -------------------------------
_r38 = final_price_table.query("date=='2024-09-08'").iloc[0]
_ctrl3 = final_price_table.query("scenario.str.startswith('3.') and pair=='대조군'",
                                 engine='python')['revenue'].mean()
print(f"\n  [반례] 강수확률 50% 경기(2024-09-08 일)의 수익 {_r38['revenue']:,}원은")
print(f"         맑은 날 대조군 평균 {_ctrl3:,.0f}원보다 오히려 높다.")
print(f"         9월 일요일 달력효과 {_r38['cal']:+.2f}가 강수충격 {_r38['event']:+.4f}를")
print("         압도했기 때문이다. 같은 표 안의 절대수익끼리도 달력이 다르면 비교할 수 없다.")

M0 테이블: (10, 3) = 10팀 x 1시즌 (2024)
Capacity 테이블: (10, 3) = 10팀 x 1시즌 (2024)
[시나리오 2] WAR_gap 5.0 = 주전 5.0 - 백업 0.0
  식(2-1) WAR_lost = 5.0 × 46.2591/144 = 1.6062승
  식(2-2) Delta_W = -5.0/144 = -0.034722; 부상 후 검산승률 0.5146
  식(2-3) A(d) = 0.75 + (1-0.75)×2^(-d/8.0)
    d= 2  A=0.9602  delta=-0.04138  수요변화=-4.05%
    d=14  A=0.8243  delta=-0.03552  수요변화=-3.49%
    d=30  A=0.7686  delta=-0.03312  수요변화=-3.26%
    d=45  A=0.7551  delta=-0.03254  수요변화=-3.20%
    d=58  A=0.7516  delta=-0.03239  수요변화=-3.19%
  ※ rho=0.75는 직접 추정치가 아닌 구조적 가정이며 [A-7]의 rho×반감기 격자로 검증한다.
[공통] 홈팀 Jaguars  M0 = 17,575   capacity = 23,750
 [A-12] 표 1. 최종 가격 제안표 — 기본가 일반석 12,000원 / 프리미엄석 35,000원
         scenario pair       date day_of_week   cal     event  log_shock   M_t  p_D7_G  p_D7_G_x  p_D7_P  p_D7_P_x  p_D3_G  p_D3_G_x  p_D3_P  p_D3_P_x  audience   revenue  인용가능
   1. 긍정 (스타 대기록)  대조군 2024-04-05           금  0.10  0.000000     0.1000 19423   15651      1.30   39761      1.14   16830      1.40   42214      1.21    